In [1]:
import os
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
# 参数列表
CONFIG = {
    # 信号参数
    'sps':  4,
    'signal_length':  1024,
    'rolloff': 0.35,
    'rrc_taps': 101,


    # 数据规模
    'n_per_class_per_snr': 10,
    'snr_list_db': list(range(-2, 38, 2)),
    'train_ratio': 0.8,
    'save_dir': './sionna_dataset',

    # 信道
    'channel_type':  'mixed',
    'mixed_weights': {
        'awgn': 0.30,
        'flat_rayleigh': 0.20,
        'flat_rician': 0.15,
        'multipath': 0.35
    },

    # 发射端损伤
    'pa_enable': True,
    'pa_ibo_db': 8.0,
    'pa_p': 3,  # Rapp 模型平滑因子

    # 接收端损伤
    'impairments_enable':   True,
    'cfo_max_norm':         0.001, # 最大归一化频偏 Δf/fs
    'phase_noise_std':      0.010, # Wiener 过程 σ (rad/sample)
    'iq_gain_imb_max_db':   1.0,   # 最大 IQ 增益不平衡 (dB)
    'iq_phase_imb_max_deg': 5.0,   # 最大 IQ 相位不平衡 (°)
    'dc_offset_max':        0.020, # 最大 DC 偏移 (相对值)
    'timing_offset_max':    0.4,   # 最大分数采样偏移 (样点)

    # 调制类别
    'classes': ['BPSK', 'QPSK', 'OQPSK', '2FSK', '4FSK', '16QAM', '64QAM', 'Pi4DQPSK', 'GMSK']

    # 可复现性
}
SIG_LEN = CONFIG['signal_length']
SPS = CONFIG['sps']
NUM_SYM = SIG_LEN // SPS
NUM_CLS = len(CONFIG['classes'])

In [28]:
# 2. RRC 脉冲成形滤波器 
def design_rrc(ntaps, sps, beta):
    """Root Raised Cosine 滤波器（教科书标准公式）"""
    T = float(sps)
    n = np.arange(-(ntaps // 2), ntaps // 2 +1, dtype=np.float64)
    h = np.zeros_like(n)
    for i, t in enumerate(n):
        if abs(t) < 1e-12:
            h[i] = 1.0 - beta + 4.0 * beta / np.pi
        elif abs(abs(t) - T / (4.0 * beta)) < 1e-12:
            h[i] = beta / np.sqrt(2) * (
                (1 + 2 / np.pi) * np.sin(np.pi / (4 * beta)) +
                (1 - 2 / np.pi) * np.cos(np.pi / (4 * beta)))
        else:
            num = (np.sin(np.pi * t / T * (1 - beta)) + 
                   4 * beta * t / T * np.cos(np.pi * t / T * (1 + beta)))
            den = np.pi * t / T * (1 - (4 * beta * t / T) ** 2)
            h[i] = num / den
    return (h / np.linalg.norm(h)).astype(np.float64)

_RRC = design_rrc(CONFIG['rrc_taps'], SPS, CONFIG['rolloff'])

In [29]:
# 3. FFT 批量卷积（替代逐样本 np.convolve）  
def batch_fft_convolve(signals, filt):
    """
    矢量化 FFT 卷积：一次处理整个 batch

    Args:
        signals: (N, L) complex/real
        filt:    (K,)   滤波器系数
    Returns:
        (N, L)  mode='same'
    """
    N, L = signals.shape
    K = len(filt)
    fft_len = 1
    while fft_len < L + K -1:
        fft_len <<= 1
    X = np.fft.fft(signals, n=fft_len, axis=1)
    H = np.fft.fft(filt, n=fft_len)
    Y = np.fft.ifft(X * H[np.newaxis, :], axis=1)
    start = (K - 1) // 2
    return Y[:, start:start + L]

# 数字调制信号星座图分类全表

---

## 一、✅ 有星座图的调制（线性调制）

> **核心特征**：信息映射到复平面上的**离散点**（幅度和/或相位），符号之间相互独立。

### 1. PSK 族（相移键控）

| 调制 | 星座点数 | 星座特征 | 典型应用 |
|:-----|:------:|:---------|:---------|
| **BPSK** | 2 | 实轴上 ±1 | 低速率、高可靠链路 |
| **QPSK** | 4 | 单位圆上 4 点（90°间隔） | 卫星、LTE、Wi-Fi |
| **OQPSK** | 4 | 同 QPSK（Q 路延迟半符号） | CDMA、ZigBee |
| **π/4-DQPSK** | 8 | 两组 QPSK 交替旋转 45° | TETRA、PDC |
| **8PSK** | 8 | 单位圆上 8 点（45°间隔） | EDGE、DVB-S2 |
| **16PSK** | 16 | 单位圆上 16 点 | 理论研究（实际少用） |
| **DBPSK** | 2 | 同 BPSK（差分编码） | 蓝牙 BR |
| **DQPSK** | 4 | 同 QPSK（差分编码） | DECT |
| **D8PSK** | 8 | 同 8PSK（差分编码） | 蓝牙 EDR |

```
BPSK          QPSK           8PSK
                ·
 ·     ·      · · ·        ·  ·  ·
                ·          ·      ·
                            ·  ·  ·
```

### 2. QAM 族（正交幅度调制）

| 调制 | 星座点数 | 星座形状 | 典型应用 |
|:-----|:------:|:---------|:---------|
| **4QAM** | 4 | 2×2 方格 = QPSK | — |
| **16QAM** | 16 | 4×4 方格 | LTE、Wi-Fi |
| **32QAM** | 32 | 6×6 去 4 角（十字形） | DVB-T2 |
| **64QAM** | 64 | 8×8 方格 | LTE、Wi-Fi 5 |
| **128QAM** | 128 | 12×12 取最近 128 点 | 有线电视 |
| **256QAM** | 256 | 16×16 方格 | Wi-Fi 5、DOCSIS |
| **512QAM** | 512 | 十字形 | 微波回传 |
| **1024QAM** | 1024 | 32×32 方格 | Wi-Fi 6 |
| **2048QAM** | 2048 | 十字形 | 微波链路 |
| **4096QAM** | 4096 | 64×64 方格 | Wi-Fi 7 |

```
4QAM           16QAM              64QAM
·   ·       · · · ·          · · · · · · · ·
·   ·       · · · ·          · · · · · · · ·
            · · · ·          · · · · · · · ·
            · · · ·          · · · · · · · ·
                             · · · · · · · ·
                             · · · · · · · ·
                             · · · · · · · ·
                             · · · · · · · ·
```

### 3. ASK 族（幅移键控）

| 调制 | 星座点数 | 星座特征 | 典型应用 |
|:-----|:------:|:---------|:---------|
| **OOK (2ASK)** | 2 | 实轴上 0 和 1 | 光通信、RFID |
| **4ASK** | 4 | 实轴上 4 个幅度 | PAM-4（以太网） |
| **8ASK (PAM-8)** | 8 | 实轴上 8 个幅度 | 400G 光通信 |
| **16ASK (PAM-16)** | 16 | 实轴上 16 个幅度 | 理论研究 |

```
OOK:      ·        ·
          0        1

4ASK:   ·    ·    ·    ·
       -3   -1   +1   +3
```

### 4. APSK 族（幅度相位键控）

| 调制 | 星座点数 | 星座特征 | 典型应用 |
|:-----|:------:|:---------|:---------|
| **16APSK** | 16 | 2 环（4+12） | DVB-S2 |
| **32APSK** | 32 | 3 环（4+12+16） | DVB-S2 |
| **64APSK** | 64 | 4 环 | DVB-S2X |
| **128APSK** | 128 | 多环 | DVB-S2X |
| **256APSK** | 256 | 多环 | DVB-S2X |

```
16APSK                  32APSK
     ·  ·  ·                ·  ·  ·  ·
   ·  ·    ·  ·          ·  ·  ·    ·  ·
  ·    ··    ·          ·   ·  ··  ·   ·
   ·  ·    ·  ·          ·  ·  ·    ·  ·
     ·  ·  ·                ·  ·  ·  ·
  内环4点+外环12点       4+12+16 三环
```

> APSK 是卫星通信的首选，因为多环结构对 PA 非线性更友好。

### 5. 其他有星座图的调制

| 调制 | 说明 | 典型应用 |
|:-----|:-----|:---------|
| **Star-QAM** | 多环等间隔排列 | 光通信 |
| **Hex-QAM** | 六边形密排 | 理论研究 |
| **CAP (无载波幅度相位)** | 类似 QAM，不同实现 | DSL |
| **Probabilistic Shaped QAM** | 概率整形 QAM | 5G NR、光通信 |

---

## 二、❌ 没有传统星座图的调制（非线性/频率调制）

> **核心特征**：信息承载在**频率**或**连续相位轨迹**中，当前符号的输出依赖于历史状态，无法用复平面上的固定离散点表示。

### 1. FSK 族（频移键控）

| 调制 | 频率数 | 特征 | 典型应用 |
|:-----|:-----:|:-----|:---------|
| **2FSK** | 2 | 两频率切换 | 业余无线电、传感器 |
| **4FSK** | 4 | 四频率切换 | DMR、dPMR |
| **8FSK** | 8 | 八频率切换 | 少见 |
| **GFSK** | 2 | 高斯滤波 FSK | 蓝牙 BR、LoRa |
| **AFSK** | 2 | 音频 FSK | AX.25、APRS |

```
频率
  ↑
f₁ ─── ████      ████      ████
  |         ████      ████
f₀ ────                          
  +────────────────────────→ 时间
       "1"  "0"  "1"  "0"  "1"
```

### 2. CPM 族（连续相位调制）

| 调制 | 特征 | 典型应用 |
|:-----|:-----|:---------|
| **MSK** | 调制指数 h=0.5 的 CPFSK | GSM（变体 GMSK） |
| **GMSK** | 高斯滤波 MSK | GSM、AIS |
| **CPFSK** | 连续相位 FSK (通用) | 军事通信 |
| **Multi-h CPM** | 多调制指数轮替 | 军事、深空 |
| **Partial Response CPM** | 相位响应跨多符号 | 军事抗干扰 |
| **SOQPSK** | 成形偏移 QPSK（本质是 CPM） | 遥测（IRIG-106） |

```
相位
  ↑    ╱╲
  |   ╱  ╲    ╱╲
  |  ╱    ╲  ╱  ╲     ← 相位连续变化，不跳变
  | ╱      ╲╱    ╲
  +──────────────────→ 时间
  连续相位轨迹 — 没有固定的"点"可画
```

### 3. 扩频调制

| 调制 | 特征 | 典型应用 |
|:-----|:-----|:---------|
| **DSSS** | 直接序列扩频 | GPS、CDMA |
| **FHSS** | 跳频扩频 | 蓝牙、军事 |
| **CSS** | 啁啾扩频 | LoRa |

### 4. 脉冲调制

| 调制 | 特征 | 典型应用 |
|:-----|:-----|:---------|
| **PPM** | 脉冲位置调制 | UWB、光通信 |
| **PWM** | 脉冲宽度调制 | 控制系统 |
| **UWB-IR** | 超宽带脉冲无线电 | IEEE 802.15.4a |

---

## 三、⚠️ 灰色地带（可以画但意义不同）

| 调制 | 能画星座吗？ | 说明 |
|:-----|:----------:|:-----|
| **OQPSK** | ✅ 能 | 星座同 QPSK，但 I/Q 有半符号时延，相位轨迹不过原点 |
| **MSK** | ⚠️ 勉强 | 可视为 OQPSK + 半正弦脉冲成形，采样后可画 ±1，但本质是连续相位 |
| **GMSK** | ⚠️ 勉强 | 类似 MSK，高斯滤波后相位记忆更长 |
| **π/4-DQPSK** | ✅ 能 | 有 8 个可能的星座点，但需差分解码 |
| **OFDM** | ✅ 能 | 每个子载波上画星座（子载波用 QAM/PSK），整体不是单一星座 |
| **SC-FDMA** | ✅ 能 | 同 OFDM |
| **SOQPSK** | ⚠️ 勉强 | 名字有 QPSK，但实际是 CPM |

---

## 四、总结判断标准

```
┌───────────────────────────────────────────────────────────┐
│                 有没有星座图？判断规则                       │
│                                                           │
│   信息映射到复平面的 ──→  ✅ 有星座图                       │
│   离散固定点？         │                                   │
│        │              │  PSK, QAM, ASK, APSK              │
│        │              │  每个符号 = 一个复数                │
│        ↓              │                                   │
│       否              │                                   │
│        │              │                                   │
│   信息在频率或 ───────→  ❌ 没有传统星座图                   │
│   连续相位轨迹中？     │                                   │
│                       │  FSK, CPM, MSK, GMSK              │
│                       │  输出取决于历史状态                  │
│                       │  相位/频率连续变化                   │
└───────────────────────────────────────────────────────────┘
```

---

## 五、代码中 14 种调制的分类

| 调制 | 有星座图 | 生成方式 |
|:-----|:-------:|:---------|
| BPSK | ✅ | 查星座表 → 上采样 → RRC |
| QPSK | ✅ | 查星座表 → 上采样 → RRC |
| 8PSK | ✅ | 查星座表 → 上采样 → RRC |
| OQPSK | ✅ | 查星座表 → I/Q 错半符号 → RRC |
| Pi4DQPSK | ✅ | 差分编码 → 查星座表 → RRC |
| 16QAM | ✅ | 查星座表 → 上采样 → RRC |
| 32QAM | ✅ | 查星座表 → 上采样 → RRC |
| 64QAM | ✅ | 查星座表 → 上采样 → RRC |
| 128QAM | ✅ | 查星座表 → 上采样 → RRC |
| 256QAM | ✅ | 查星座表 → 上采样 → RRC |
| 2ASK | ✅ | 查星座表 → 上采样 → RRC |
| **2FSK** | ❌ | **直接生成频率切换波形** |
| **4FSK** | ❌ | **直接生成频率切换波形** |
| **MSK** | ❌ | **直接生成连续相位波形** |

> **这就是为什么 `CONSTELLATIONS` 字典里只有 PSK/QAM/ASK，而 FSK 和 MSK 需要在后面用完全不同的代码单独生成。**

In [ ]:
# 4.星座定义
def _make_psk(M):
    return np.exp(1j * 2 * np.pi * np.arange(M) / M)

# PSK 星座图数学公式详解

---

## 一、核心公式

$$\boxed{s_m = e^{\,j \frac{2\pi}{M} m}, \quad m = 0, 1, 2, \ldots, M-1}$$

其中：
- $M$ = 调制阶数（星座点总数）
- $m$ = 第 $m$ 个符号的索引
- $s_m$ = 第 $m$ 个星座点（复数）

---

## 二、展开为实部和虚部

由欧拉公式 $e^{j\theta} = \cos\theta + j\sin\theta$：

$$s_m = \cos\!\left(\frac{2\pi m}{M}\right) + j\,\sin\!\left(\frac{2\pi m}{M}\right)$$

即：

$$I_m = \cos\!\left(\frac{2\pi m}{M}\right), \quad Q_m = \sin\!\left(\frac{2\pi m}{M}\right)$$

---

## 三、几何意义

每个星座点在**单位圆**上：

$$|s_m| = \left|e^{\,j\frac{2\pi m}{M}}\right| = 1 \quad \text{（恒包络）}$$

相邻点的角度间隔：

$$\Delta\theta = \frac{2\pi}{M} = \frac{360°}{M}$$

---

## 四、代码逐步对应

```python
def _make_psk(M):
    return np.exp(1j * 2 * np.pi * np.arange(M) / M)
```

| 代码 | 数学 | 值（以 QPSK, M=4 为例） |
|:-----|:-----|:----------------------|
| `np.arange(M)` | $m = 0, 1, \ldots, M\!-\!1$ | `[0, 1, 2, 3]` |
| `np.arange(M) / M` | $\frac{m}{M}$ | `[0, 0.25, 0.5, 0.75]` |
| `2 * np.pi * ...` | $\frac{2\pi m}{M}$ | `[0, π/2, π, 3π/2]` |
| `1j * ...` | $j \cdot \frac{2\pi m}{M}$ | `[0, jπ/2, jπ, j3π/2]` |
| `np.exp(...)` | $e^{j\frac{2\pi m}{M}}$ | `[1, j, -1, -j]` |

---

## 五、各 PSK 的具体星座

### BPSK（M=2）

$$s_m = e^{j\pi m}, \quad m = 0, 1$$

| $m$ | 角度 | $s_m$ | I | Q |
|:---:|:----:|:-----:|:-:|:-:|
| 0 | $0°$ | $1$ | 1 | 0 |
| 1 | $180°$ | $-1$ | -1 | 0 |

```
Q
↑
|
·───────────·──→ I
-1          +1
```

### QPSK（M=4）

$$s_m = e^{j\frac{\pi}{2} m}, \quad m = 0, 1, 2, 3$$

| $m$ | 角度 | $s_m$ | I | Q |
|:---:|:----:|:-----:|:-:|:-:|
| 0 | $0°$ | $1$ | 1 | 0 |
| 1 | $90°$ | $j$ | 0 | 1 |
| 2 | $180°$ | $-1$ | -1 | 0 |
| 3 | $270°$ | $-j$ | 0 | -1 |

```
        Q
        ↑
        · (j)
        |
(-1) ·──┼──· (+1) → I
        |
        · (-j)
```

### 8PSK（M=8）

$$s_m = e^{j\frac{\pi}{4} m}, \quad m = 0, 1, \ldots, 7$$

| $m$ | 角度 | $s_m$ | I | Q |
|:---:|:----:|:-----:|:---:|:---:|
| 0 | $0°$ | $1$ | $1$ | $0$ |
| 1 | $45°$ | $\frac{1+j}{\sqrt{2}}$ | $\frac{\sqrt{2}}{2}$ | $\frac{\sqrt{2}}{2}$ |
| 2 | $90°$ | $j$ | $0$ | $1$ |
| 3 | $135°$ | $\frac{-1+j}{\sqrt{2}}$ | $-\frac{\sqrt{2}}{2}$ | $\frac{\sqrt{2}}{2}$ |
| 4 | $180°$ | $-1$ | $-1$ | $0$ |
| 5 | $225°$ | $\frac{-1-j}{\sqrt{2}}$ | $-\frac{\sqrt{2}}{2}$ | $-\frac{\sqrt{2}}{2}$ |
| 6 | $270°$ | $-j$ | $0$ | $-1$ |
| 7 | $315°$ | $\frac{1-j}{\sqrt{2}}$ | $\frac{\sqrt{2}}{2}$ | $-\frac{\sqrt{2}}{2}$ |

```
         Q
         ↑
     ·   ·   ·
      ╲  |  ╱
       ╲ | ╱
  · ────╋──── · → I
       ╱ | ╲
      ╱  |  ╲
     ·   ·   ·
```

---

## 六、PSK 的关键性质

### 1. 恒包络

$$|s_m| = 1, \quad \forall\, m$$

所有信息**只在相位中**，幅度恒定 → 对 PA 非线性不敏感。

### 2. 平均功率归一化

$$\frac{1}{M}\sum_{m=0}^{M-1}|s_m|^2 = \frac{1}{M}\sum_{m=0}^{M-1}1 = 1$$

代码无需额外归一化（已经是单位功率）。

### 3. 最小欧氏距离

相邻星座点间距：

$$d_{\min} = |s_{m+1} - s_m| = \left|e^{j\frac{2\pi(m+1)}{M}} - e^{j\frac{2\pi m}{M}}\right| = 2\sin\!\left(\frac{\pi}{M}\right)$$

| M | $\Delta\theta$ | $d_{\min}$ | 相对抗噪声能力 |
|:-:|:-:|:-:|:-:|
| 2 (BPSK) | 180° | $2.000$ | ⭐⭐⭐⭐⭐ |
| 4 (QPSK) | 90° | $\sqrt{2} \approx 1.414$ | ⭐⭐⭐⭐ |
| 8 (8PSK) | 45° | $0.765$ | ⭐⭐⭐ |
| 16 | 22.5° | $0.390$ | ⭐⭐ |
| 32 | 11.25° | $0.196$ | ⭐ |

> **M 越大 → 点越密 → $d_{\min}$ 越小 → 越容易被噪声搞错。**

### 4. 对称性

$$s_m \cdot e^{j\frac{2\pi}{M}} = s_{m+1}$$

整个星座旋转 $\frac{2\pi}{M}$ 后**与自身重合** → $M$ 阶旋转对称。

---

## 七、总结

$$\boxed{s_m = e^{\,j\frac{2\pi m}{M}} = \cos\!\left(\frac{2\pi m}{M}\right) + j\sin\!\left(\frac{2\pi m}{M}\right)}$$

```
本质：在单位圆上等间隔放 M 个点
      第 m 个点的角度 = m × (360°/M)
      所有点幅度 = 1（恒包络）
      所有信息编码在相位中
```

In [23]:
def _make_square_qam(M):
    """方形 QAM: 4, 16, 64, 256"""
    k = int(np.sqrt(M))
    assert k * k == M
    levels = 2.0 * np.arange(k) - (k - 1)
    gI, gQ = np.meshgrid(levels, levels)
    pts = (gI + 1j * gQ).flatten()
    return pts / np.sqrt(np.mean(np.abs(pts) ** 2))

def _make_cross_qam(M):
    """
    Cross-QAM: 32, 128

    32-QAM:  6×6 格点去掉 4 角 = 32
    128-QAM: 12×12 格点取功率最低 128 个
    """
    if M == 32:
        v = np.array([-5., -3., -1., 1., 3., 5.])
        gI, gQ = np.meshgrid(v, v)
        pts = (gI + 1j * gQ).flatten()
        mask = ~((np.abs(gI.flatten()) == 5) & (np.abs(gQ.flatten()) == 5))
        pts = pts[mask]
    elif M == 128:
        v = 2.0 * np.arange(12) - 11
        gI, gQ = np.meshgrid(v, v)
        pts = (gI + 1j * gQ).flatten()
        idx = np.argsort(np.abs(pts) ** 2)[:128]
        pts = pts[idx]
    else:
        raise ValueError(f"Cross-QAM 未定义 M={M}")
    return pts / np.sqrt(np.mean(np.abs(pts) ** 2))

# QAM 星座图数学公式详解

---

## 一、方形 QAM（Square QAM）

### 代码

```python
def _make_square_qam(M):
    k = int(np.sqrt(M))
    assert k * k == M
    levels = 2.0 * np.arange(k) - (k - 1)
    gI, gQ = np.meshgrid(levels, levels)
    pts = (gI + 1j * gQ).flatten()
    return pts / np.sqrt(np.mean(np.abs(pts) ** 2))
```

### 核心公式

$$\boxed{s_{m,n} = \frac{(2m - K + 1) + j\,(2n - K + 1)}{\sqrt{P_{\text{avg}}}}, \quad m, n = 0, 1, \ldots, K\!-\!1}$$

其中：
- $K = \sqrt{M}$（每边点数）
- $P_{\text{avg}}$ 为归一化前的平均功率

---

### 逐行拆解（以 16QAM, M=16, K=4 为例）

#### Step 1：计算每边点数

```python
k = int(np.sqrt(M))     # K = √16 = 4
```

$$K = \sqrt{M} = 4$$

#### Step 2：生成坐标电平

```python
levels = 2.0 * np.arange(k) - (k - 1)
```

| 代码 | 数学 | 值 |
|:-----|:-----|:---|
| `np.arange(k)` | $m = 0,1,2,3$ | `[0, 1, 2, 3]` |
| `2.0 * np.arange(k)` | $2m$ | `[0, 2, 4, 6]` |
| `... - (k-1)` | $2m - (K\!-\!1)$ | `[-3, -1, 1, 3]` |

一般公式：

$$\ell_m = 2m - (K - 1), \quad m = 0, 1, \ldots, K\!-\!1$$

**即以原点为中心、间距为 2 的等间隔电平：**

| $K$ | levels |
|:---:|:-------|
| 2 (4QAM) | $[-1, 1]$ |
| 4 (16QAM) | $[-3, -1, 1, 3]$ |
| 8 (64QAM) | $[-7, -5, -3, -1, 1, 3, 5, 7]$ |
| 16 (256QAM) | $[-15, -13, \ldots, 13, 15]$ |

#### Step 3：网格化

```python
gI, gQ = np.meshgrid(levels, levels)
```

$$(\text{gI}, \text{gQ}) = \text{所有} \;(\ell_m, \ell_n)\; \text{的组合}$$

以 16QAM 为例，生成 $4 \times 4 = 16$ 个格点：

```
Q ↑
3 │  ·   ·   ·   ·       (gI, gQ) = (-3,3) (-1,3) (1,3) (3,3)
1 │  ·   ·   ·   ·                   (-3,1) (-1,1) (1,1) (3,1)
-1│  ·   ·   ·   ·                   (-3,-1)(-1,-1)(1,-1)(3,-1)
-3│  ·   ·   ·   ·                   (-3,-3)(-1,-3)(1,-3)(3,-3)
  └──────────────── → I
    -3  -1   1   3
```

#### Step 4：组合为复数

```python
pts = (gI + 1j * gQ).flatten()
```

$$s_{m,n}^{\text{raw}} = \ell_m + j\,\ell_n$$

例如：$s_{0,0}^{\text{raw}} = -3 + j(-3) = -3-3j$

#### Step 5：功率归一化

```python
pts / np.sqrt(np.mean(np.abs(pts) ** 2))
```

归一化前平均功率：

$$P_{\text{avg}} = \frac{1}{M} \sum_{m,n} |\ell_m + j\,\ell_n|^2 = \frac{1}{M} \sum_{m,n} (\ell_m^2 + \ell_n^2)$$

由对称性可分解：

$$P_{\text{avg}} = 2 \cdot \frac{1}{K}\sum_{m=0}^{K-1} \ell_m^2$$

以 16QAM 为例：

$$\frac{1}{K}\sum \ell_m^2 = \frac{(-3)^2+(-1)^2+1^2+3^2}{4} = \frac{9+1+1+9}{4} = 5$$

$$P_{\text{avg}} = 2 \times 5 = 10$$

归一化后：

$$s_{m,n} = \frac{\ell_m + j\,\ell_n}{\sqrt{10}}$$

一般 $K \times K$ 方形 QAM 的平均功率公式：

$$\boxed{P_{\text{avg}} = \frac{2(M - 1)}{3}}$$

| M | $P_{\text{avg}}$ | $\sqrt{P_{\text{avg}}}$ |
|:-:|:-:|:-:|
| 4 | 2 | 1.414 |
| 16 | 10 | 3.162 |
| 64 | 42 | 6.481 |
| 256 | 170 | 13.038 |

---

### 16QAM 归一化后的完整星座

$$s_{m,n} = \frac{(2m-3) + j(2n-3)}{\sqrt{10}}, \quad m,n \in \{0,1,2,3\}$$

```
Q ↑
  │  ·       ·       ·       ·
  │ -3+3j  -1+3j   1+3j   3+3j       ÷ √10
  │
  │  ·       ·       ·       ·
  │ -3+1j  -1+1j   1+1j   3+1j       ÷ √10
  │
  │  ·       ·       ·       ·
  │ -3-1j  -1-1j   1-1j   3-1j       ÷ √10
  │
  │  ·       ·       ·       ·
  │ -3-3j  -1-3j   1-3j   3-3j       ÷ √10
  └────────────────────────────→ I
```

---

## 二、十字形 QAM（Cross QAM）

### 为什么需要十字形？

$M = 32, 128$ 不是完全平方数（$\sqrt{32} \notin \mathbb{Z}$），无法构成正方形网格。

解决方案：**从更大的正方形中去掉角落的点。**

---

### 32QAM

```python
if M == 32:
    v = np.array([-5., -3., -1., 1., 3., 5.])
    gI, gQ = np.meshgrid(v, v)
    pts = (gI + 1j * gQ).flatten()
    mask = ~((np.abs(gI.flatten()) == 5) & (np.abs(gQ.flatten()) == 5))
    pts = pts[mask]
```

#### Step 1：$6 \times 6$ 方格（36 个点）

$$v = [-5, -3, -1, 1, 3, 5]$$

```
Q ↑
5 │  ×   ·   ·   ·   ·   ×     ← × 是要去掉的角
3 │  ·   ·   ·   ·   ·   ·
1 │  ·   ·   ·   ·   ·   ·
-1│  ·   ·   ·   ·   ·   ·
-3│  ·   ·   ·   ·   ·   ·
-5│  ×   ·   ·   ·   ·   ×     ← × 是要去掉的角
  └──────────────────────── → I
    -5  -3  -1   1   3   5
```

#### Step 2：去掉 4 个角

去除条件：$|I| = 5$ **且** $|Q| = 5$

$$\text{去掉：} (\pm 5, \pm 5) \quad \text{共 4 个点}$$

$$36 - 4 = 32 \quad ✓$$

```
Q ↑
5 │       ·   ·   ·   ·         ← 角落空了
3 │  ·    ·   ·   ·   ·   ·
1 │  ·    ·   ·   ·   ·   ·
-1│  ·    ·   ·   ·   ·   ·
-3│  ·    ·   ·   ·   ·   ·
-5│       ·   ·   ·   ·         ← 角落空了
  └──────────────────────── → I
    -5  -3  -1   1   3   5
```

#### 数学描述

$$\boxed{S_{32} = \big\{(\ell_m, \ell_n) : \ell \in \{-5,-3,-1,1,3,5\}\big\} \;\setminus\; \big\{(\pm5, \pm5)\big\}}$$

最后功率归一化：

$$s = \frac{I + jQ}{\sqrt{P_{\text{avg}}}}, \quad P_{\text{avg}} = \frac{1}{32}\sum_{(I,Q) \in S_{32}} (I^2 + Q^2)$$

---

### 128QAM

```python
elif M == 128:
    v = 2.0 * np.arange(12) - 11
    gI, gQ = np.meshgrid(v, v)
    pts = (gI + 1j * gQ).flatten()
    idx = np.argsort(np.abs(pts) ** 2)[:128]
    pts = pts[idx]
```

#### Step 1：$12 \times 12$ 方格（144 个点）

$$v = [-11, -9, -7, -5, -3, -1, 1, 3, 5, 7, 9, 11]$$

#### Step 2：按功率排序，取最近的 128 个

$$\text{对所有 144 个点，按} \; |s|^2 = I^2 + Q^2 \; \text{从小到大排序}$$

$$\text{保留前 128 个（功率最小的），丢弃 16 个角落点}$$

$$144 - 16 = 128 \quad ✓$$

```
      · · · · · · · · · ·
    · · · · · · · · · · · ·
    · · · · · · · · · · · ·
  · · · · · · · · · · · · · ·       ← 不是这样！
```

实际形状（近似）：

```
Q ↑
  │        · · · · · · · ·
  │      · · · · · · · · · ·
  │    · · · · · · · · · · · ·
  │    · · · · · · · · · · · ·
  │  · · · · · · · · · · · · · ·     ← 不对...
```

更准确地说，去掉的是四个角上离原点最远的 16 个点：

```
Q ↑         (12×12 去掉 16 个角)
  │         · · · · · · · ·
  │       · · · · · · · · · ·
  │     · · · · · · · · · · · ·
  │     · · · · · · · · · · · ·
  │     · · · · · · · · · · · ·
  │     · · · · · · · · · · · ·
  │     · · · · · · · · · · · ·
  │     · · · · · · · · · · · ·
  │     · · · · · · · · · · · ·
  │     · · · · · · · · · · · ·
  │       · · · · · · · · · ·
  │         · · · · · · · ·
  └──────────────────────────→ I
              十字形
```

#### 数学描述

$$\boxed{S_{128} = \underset{|S|=128}{\arg\min} \sum_{s \in S} |s|^2, \quad s \in \{(2m\!-\!11) + j(2n\!-\!11)\}_{m,n=0}^{11}}$$

即：从 $12 \times 12$ 方格中选出**平均功率最小**的 128 个点。

---

## 三、方形 vs 十字形对比

| | 方形 QAM | 十字形 QAM |
|:--|:---------|:-----------|
| **适用 M** | $M = K^2$（完全平方数） | $M \ne K^2$ |
| **例子** | 4, 16, 64, 256 | 32, 128 |
| **构造方法** | $K \times K$ 网格 | 大方格去角 |
| **最小距离** | 均匀 | 均匀（点间距相同） |
| **形状** | 正方形 | 十字形 / 圆角方形 |
| **公式** | $s = \ell_m + j\ell_n$ | $s \in \text{Grid} \setminus \text{Corners}$ |

---

## 四、QAM 关键性质

### 1. 非恒包络

$$|s_{m,n}| = \frac{\sqrt{\ell_m^2 + \ell_n^2}}{\sqrt{P_{\text{avg}}}} \quad \text{（不同点幅度不同）}$$

以 16QAM 为例：

| 位置 | 原始幅度 | 归一化幅度 | 占比 |
|:----:|:-------:|:---------:|:----:|
| 内圈 4 点 $(\pm1,\pm1)$ | $\sqrt{2}$ | $\sqrt{2/10}$ | 25% |
| 中圈 8 点 $(\pm1,\pm3)$ 等 | $\sqrt{10}$ | $1$ | 50% |
| 外圈 4 点 $(\pm3,\pm3)$ | $\sqrt{18}$ | $\sqrt{18/10}$ | 25% |

**→ 外圈点对 PA 非线性敏感（PAPR 问题）**

### 2. 最小欧氏距离

点间距（归一化前）固定为 $d = 2$，归一化后：

$$\boxed{d_{\min} = \frac{2}{\sqrt{P_{\text{avg}}}} = \sqrt{\frac{6}{M-1}}}$$

| M | $d_{\min}$ | 对比 PSK 的 $d_{\min}$ |
|:-:|:----------:|:-----:|
| 4 | 1.155 | QPSK: 1.414 → **PSK 更好** |
| 16 | 0.632 | 16PSK: 0.390 → **QAM 更好** |
| 64 | 0.309 | 64PSK: 0.098 → **QAM 远优** |
| 256 | 0.153 | 256PSK: 0.025 → **QAM 碾压** |

> **规律：$M \le 4$ 时 PSK ≥ QAM；$M \ge 8$ 时 QAM 远优于 PSK。**

### 3. 功率归一化的意义

$$\frac{1}{M}\sum_{m=0}^{M-1} |s_m|^2 = 1$$

保证**不同阶数的 QAM 在相同 SNR 下公平比较**。

---

## 五、总结公式卡片

$$\boxed{
\begin{aligned}
&\textbf{方形 QAM}(M = K^2): \\
&\quad \ell_m = 2m - (K\!-\!1), \quad m = 0,\ldots,K\!-\!1 \\
&\quad s_{m,n}^{\text{raw}} = \ell_m + j\,\ell_n \\
&\quad s_{m,n} = \frac{s_{m,n}^{\text{raw}}}{\sqrt{\frac{2(M-1)}{3}}} \\[12pt]
&\textbf{十字形 QAM}(M = 32, 128): \\
&\quad \text{从更大方格中去掉角落点，使} |S| = M \\
&\quad \text{同样功率归一化到} \frac{1}{M}\sum|s|^2 = 1
\end{aligned}
}$$

# 星座定义在整个流程中的角色

---

## 一、为什么星座定义在这个位置？

整个代码的结构是 **"先准备工具，后组装流水线"**：

```
┌─────────────────────────────────────────────────────┐
│  1. 配置参数 CONFIG              ← 全局设定          │
│  2. RRC 脉冲成形滤波器           ← 工具：滤波器系数   │
│  3. FFT 批量卷积                 ← 工具：快速卷积函数  │
│  4. 星座定义 ★                   ← 工具：符号字母表   │
│  5. 信道模型                     ← 工具：信道类       │
│  6. RF 损伤                      ← 工具：损伤类       │
│  7. 调制信号生成（后面的代码）     ← 正式组装流水线    │
│     ↑ 调用上面所有工具             │
└─────────────────────────────────────────────────────┘
```

**星座定义不是在"生成信号"，而是在"准备字母表"。**

类比：
```
星座定义  =  准备好 26 个英文字母的字模
信号生成  =  用这些字模排版、印刷、装订成书
```

---

## 二、星座定义完成了调制的哪部分工作？

调制信号的完整生成流程：

```
Step 1: 星座定义           ← 第4节做的事（仅此而已）
        "QPSK有4个点，分别在哪里"

Step 2: 随机选符号          ← 第7节（信号生成）
        从星座中随机选 n_sym 个点
        
Step 3: 脉冲成形           ← 第7节调用第2、3节
        上采样 + RRC 滤波
        
Step 4: PA 非线性          ← 第7节调用第6节
        
Step 5: 信道衰落 + AWGN    ← 第7节调用第5节
        
Step 6: RX 损伤            ← 第7节调用第6节
```

**星座定义只完成了 Step 1：定义每种调制的符号集合（字母表）。**

---

## 三、哪些调制有星座图，哪些没有？

### ✅ 有星座图的调制（幅度/相位映射）

| 调制类型 | 星座点数 | 星座特征 | 代码中定义方式 |
|:---------|:------:|:---------|:-------------|
| BPSK | 2 | 实轴上两个点 | `_make_psk(2)` |
| QPSK | 4 | 正方形 4 个角 | `_make_psk(4)` |
| 8PSK | 8 | 圆上等间隔 8 点 | `_make_psk(8)` |
| OQPSK | 4 | 同 QPSK（Q 路延迟半符号） | 基于 QPSK 星座 |
| Pi/4-DQPSK | 8 | 两组旋转 QPSK 交替 | 差分编码逻辑 |
| 16QAM | 16 | 4×4 方格 | `_make_square_qam(16)` |
| 32QAM | 32 | 6×6 去 4 角 (十字形) | `_make_cross_qam(32)` |
| 64QAM | 64 | 8×8 方格 | `_make_square_qam(64)` |
| 128QAM | 128 | 12×12 取最近 128 个 | `_make_cross_qam(128)` |
| 256QAM | 256 | 16×16 方格 | `_make_square_qam(256)` |
| 2ASK | 2 | 实轴上两个幅度 | 可用星座，但通常单独实现 |

### ❌ 没有传统星座图的调制（频率/连续相位映射）

| 调制类型 | 为什么没有固定星座 | 信号生成方式 |
|:---------|:------------------|:------------|
| **2FSK** | 信息在**频率**中，不在幅度/相位 | 切换两个频率的正弦波 |
| **4FSK** | 同上，4 个频率 | 切换四个频率的正弦波 |
| **MSK** | **连续相位**调制，相位轨迹连续 | 相位连续累积，不是跳变 |

```
有星座图的调制                    没有星座图的调制

Q ↑    · · · ·                    频率
  |    · · · ·                    ↑  ─── f₁ ───    ─── f₁ ───
  |    · · · ·                    |           ─── f₀ ───
  +--------→ I                    +──────────────────────→ 时间
     16QAM: 符号 = 复平面上的点       FSK: 符号 = 频率的切换
```

---

## 四、为什么需要单独定义星座？

### 原因 1：分离 "是什么" 和 "怎么用"

```python
# 星座定义：只回答 "16QAM 的 16 个点在哪里"
CONSTELLATIONS = {
    '16QAM': _make_square_qam(16),   # 一次性计算好
    ...
}

# 信号生成：回答 "怎么用这些点生成一段信号"
symbols = CONSTELLATIONS['16QAM'][random_indices]  # 取符号
signal = fft_convolve(upsample(symbols), rrc)      # 脉冲成形
```

### 原因 2：预计算效率

```python
# ❌ 不预定义：每次生成信号都要重新算星座
for snr in snr_list:                    # 20 个 SNR
    for cls in classes:                  # 14 种调制
        for batch in range(many):
            constellation = _make_square_qam(64)    # 重复计算！
            symbols = constellation[...]

# ✅ 预定义：程序启动时算一次，之后直接查表
CONSTELLATIONS = { ... }                # 只算 1 次
for snr in snr_list:
    for cls in classes:
        for batch in range(many):
            symbols = CONSTELLATIONS['64QAM'][...]  # 直接查表
```

### 原因 3：FSK/MSK 需要完全不同的生成逻辑

```python
# PSK/QAM 的生成逻辑：
#   选符号 → 上采样 → RRC滤波
#   统一流程，区别只在星座点

# FSK 的生成逻辑：
#   选频率 → 生成连续正弦波 → 不需要 RRC
#   完全不同的流程，根本没有"星座"可查

# MSK 的生成逻辑：
#   连续相位累积 → 保证相位连续
#   也不是从星座查表
```

所以代码后面的信号生成部分一定是这样的结构：

```python
if modulation in ['BPSK','QPSK','8PSK','16QAM',...]:
    symbols = _rand_symbols(modulation, N)     # ← 查星座表
    signal = pulse_shape(symbols)              # ← RRC 成形
    
elif modulation in ['2FSK', '4FSK']:
    signal = generate_fsk(...)                 # ← 完全不同的生成方式
    
elif modulation == 'MSK':
    signal = generate_msk(...)                 # ← 又一种不同的方式
```

---

## 五、总结图

```
                    ┌──────────────────────────┐
                    │     星座定义 (第4节)       │
                    │  "每种调制的符号字母表"     │
                    │                          │
                    │  BPSK:  {·, ·}           │
                    │  QPSK:  {·,·,·,·}        │
                    │  16QAM: {·× 16}          │
                    │  ...                     │
                    │                          │
                    │  FSK/MSK: ❌ 不在这里定义  │
                    └────────────┬─────────────┘
                                │ 被引用
                                ▼
┌────────────────────────────────────────────────────┐
│              信号生成 (第7节)                        │
│                                                    │
│  PSK/QAM 类:  查表 → 选符号 → 上采样 → RRC → ...   │
│  FSK 类:      直接生成频率切换波形                   │
│  MSK 类:      直接生成连续相位波形                   │
│                    │                               │
│                    ▼                               │
│             PA → 信道 → 损伤 → 输出                 │
└────────────────────────────────────────────────────┘
```

> **一句话：星座定义 = 提前准备好的"字母表"，只有 PSK/QAM 类调制需要它；FSK/MSK 的信息承载在频率/相位轨迹上，没有固定星座，需要完全不同的生成逻辑。**

In [24]:
CONSTELLATIONS = {
    'BPSK':   _make_psk(2),
    'QPSK':   _make_psk(4),
    '8PSK':   _make_psk(8),
    '16QAM':  _make_square_qam(16),
    '32QAM':  _make_cross_qam(32),
    '64QAM':  _make_square_qam(64),
    '128QAM': _make_cross_qam(128),
    '256QAM': _make_square_qam(256),
}

def _rand_symbols(name, n_batch, n_sym=NUM_SYM):
    """从指定星座随机选符号 → (n_batch, n_sym) complex"""
    # n_batch 生成多少条独立信号
    # n_sym 每条信号包含多少个符号 signal_length(默认1024)➗sps(默认4) = 256
    c = CONSTELLATIONS[name]
    return c[np.random.randint(0, len(c), (n_batch, n_sym))]

# `c[np.random.randint(0, len(c), (n_batch, n_sym))]` 详解

## 从内到外，逐层拆解

---

### 第 1 层：`c`

```python
c = CONSTELLATIONS['QPSK']
# c = [1+0j,  0+1j,  -1+0j,  0-1j]
#      c[0]    c[1]    c[2]    c[3]
```

就是一个**长度为 M 的一维数组**，存着所有星座点。

---

### 第 2 层：`len(c)`

```python
len(c)  # → 4（QPSK 有 4 个星座点）
```

---

### 第 3 层：`np.random.randint(0, len(c), (n_batch, n_sym))`

```python
np.random.randint(0, 4, (3, 5))
#                 ↑  ↑   ↑
#                低 高  形状
#          从[0,4)中随机取整数，生成 3×5 的矩阵
```

结果示例：

```python
idx = [[2, 0, 3, 1, 0],
       [1, 1, 0, 3, 2],
       [0, 3, 2, 2, 1]]

# shape = (3, 5)
# 每个值 ∈ {0, 1, 2, 3}
```

---

### 第 4 层：`c[idx]` —— **数组索引取值**

这是 NumPy 的**花式索引（Fancy Indexing）**：

```
c = [1+0j,  0+1j,  -1+0j,  0-1j]
      ↑[0]   ↑[1]    ↑[2]   ↑[3]
```

**用 idx 中的每个数字去 c 里查表：**

```
idx:                              c[idx]:

[[2, 0, 3, 1, 0],        [[ c[2],  c[0],  c[3],  c[1],  c[0] ],
 [1, 1, 0, 3, 2],    →    [ c[1],  c[1],  c[0],  c[3],  c[2] ],
 [0, 3, 2, 2, 1]]         [ c[0],  c[3],  c[2],  c[2],  c[1] ]]
```

代入具体值：

```
idx:                              c[idx]:

[[2, 0, 3, 1, 0],        [[-1,    1,   -j,    j,    1 ],
 [1, 1, 0, 3, 2],    →    [  j,    j,    1,   -j,   -1 ],
 [0, 3, 2, 2, 1]]         [  1,   -j,   -1,   -1,    j ]]

                          shape = (3, 5) ← 和 idx 形状相同
```

---

## 用最简单的例子理解

```python
c = ['苹果', '香蕉', '橙子']
#     c[0]    c[1]    c[2]

idx = [2, 0, 1, 2, 0]

c[idx] = ['橙子', '苹果', '香蕉', '橙子', '苹果']
#          c[2]    c[0]    c[1]    c[2]    c[0]
```

> **idx 说"我要第几个"，c[idx] 就把对应的东西取出来。**

升级到二维：

```python
idx = [[2, 0],
       [1, 2]]

c[idx] = [['橙子', '苹果'],    ← 第0行: c[2], c[0]
          ['香蕉', '橙子']]    ← 第1行: c[1], c[2]
```

> **idx 是什么形状，输出就是什么形状。**

---

## 图示总结

```
                    星座字母表 c (长度 M)
                    ┌────┬────┬────┬────┐
                    │c[0]│c[1]│c[2]│c[3]│
                    │ 1  │  j │ -1 │ -j │
                    └──┬─┴──┬─┴──┬─┴──┬─┘
                       │    │    │    │
    随机索引矩阵 idx    │    │    │    │
    ┌───┬───┬───┐      │    │    │    │
    │ 2 │ 0 │ 3 │──────┼────┘    │    │
    ├───┼───┼───┤      │         │    │
    │ 1 │ 1 │ 0 │──────┘         │    │
    ├───┼───┼───┤                 │    │
    │ 0 │ 3 │ 2 │────────────────┘    │
    └───┴───┴───┘                     │
         ↓ 查表                        │
    ┌────┬────┬────┐                   │
    │ -1 │  1 │ -j │  ← c[2],c[0],c[3]│
    ├────┼────┼────┤                   │
    │  j │  j │  1 │  ← c[1],c[1],c[0]│
    ├────┼────┼────┤                   │
    │  1 │ -j │ -1 │  ← c[0],c[3],c[2]│
    └────┴────┴────┘
    输出: 和 idx 同形状
```

---

## 一句话总结

> **`c[idx]`：把 idx 矩阵里的每个数字当作下标，去 c 数组里取对应的星座点，输出形状与 idx 相同。**

# `_rand_symbols` 函数详解

## 一句话概括

> **模拟"随机比特数据经过星座映射"的过程，生成一批随机调制符号。**

---

## 逐行拆解

```python
def _rand_symbols(name, n_batch, n_sym=NUM_SYM):
    c = CONSTELLATIONS[name]
    return c[np.random.randint(0, len(c), (n_batch, n_sym))]
```

| 行 | 代码 | 作用 | 示例（name='QPSK', n_batch=3, n_sym=5） |
|:--:|:-----|:-----|:------|
| 1 | `c = CONSTELLATIONS[name]` | 查星座表，取出字母表 | `c = [1, j, -1, -j]`（4个点） |
| 2 | `np.random.randint(0, len(c), ...)` | 随机生成索引矩阵 | shape=(3,5) 的矩阵，值∈{0,1,2,3} |
| 3 | `c[索引矩阵]` | 用索引从星座中取符号 | shape=(3,5) 的复数矩阵 |

---

## 具体例子

### QPSK, n_batch=3, n_sym=5

```
Step 1: 查表
c = [1, j, -1, -j]     ← 4个星座点

Step 2: 随机索引
idx = [[2, 0, 3, 1, 0],
       [1, 1, 0, 3, 2],
       [0, 3, 2, 2, 1]]     ← 每个值 ∈ {0,1,2,3}

Step 3: 索引取符号
output = c[idx] = 
      [[-1,  1, -j,  j,  1],      ← 第0条信号的5个符号
       [ j,  j,  1, -j, -1],      ← 第1条信号的5个符号
       [ 1, -j, -1, -1,  j]]      ← 第2条信号的5个符号

shape = (3, 5)  即 (n_batch, n_sym)
```

---

## 对应真实通信系统的哪一步？

```
真实系统:
  随机比特流 → 星座映射 → 调制符号
  [0,1,1,0]    查表      -1+1j

本函数:
  跳过比特    直接随机选星座点   → 调制符号
              （效果等价）
```

详细对比：

```
┌─────────────────────────────────────────────────┐
│  真实系统                                        │
│                                                  │
│  比特流: [0,1,1,0,0,1,...]                       │
│           │                                      │
│           ▼ 每log₂(M)个比特一组                   │
│  分组:   [01] [10] [01] ...     (QPSK: 2bit/组)  │
│           │                                      │
│           ▼ 查星座映射表                           │
│  符号:   [ j] [-1] [ j] ...                      │
│                                                  │
├─────────────────────────────────────────────────┤
│  本函数（等效简化）                                │
│                                                  │
│  直接: np.random.randint(0, 4, ...)              │
│           │                                      │
│           ▼ 随机选星座点                           │
│  符号:   [ j] [-1] [ j] ...                      │
│                                                  │
│  ★ 因为比特本身就是随机的，                        │
│    随机选索引 ≡ 随机比特→映射                      │
└─────────────────────────────────────────────────┘
```

---

## 为什么可以跳过比特直接选符号？

因为在**调制识别**任务中：

| 关注点 | 是否需要比特？ |
|:-------|:-------------|
| 原始比特内容是什么 | ❌ 不关心 |
| 每个符号是哪个星座点 | ❌ 不关心 |
| 信号的**统计特征**（频谱、包络、相位轨迹） | ✅ 只关心这个 |

> 随机选符号产生的**信号统计特性**与随机比特经过正规映射产生的**完全一致**。

---

## 输出去往何处？

```
_rand_symbols('16QAM', 1000, 256)
       │
       ▼  shape = (1000, 256) 复数矩阵
       │
       ▼  上采样 (插零)
       │
       ▼  RRC 脉冲成形 (卷积)
       │
       ▼  PA 非线性
       │
       ▼  信道 + AWGN
       │
       ▼  RX 损伤
       │
       ▼  最终信号样本 → 送入神经网络训练
```

---

## 总结

$$\boxed{\texttt{\_rand\_symbols} = \text{从星座字母表中随机抽取符号，模拟随机数据传输}}$$

| 参数 | 含义 |
|:-----|:-----|
| `name` | 调制类型名（决定从哪个星座取） |
| `n_batch` | 生成多少条独立信号 |
| `n_sym` | 每条信号包含多少个符号 |
| **返回值** | `(n_batch, n_sym)` 的复数矩阵 |

In [20]:
# 5.信道模型
class ChannelModel:
    def __init__(self, cfg=CONFIG):
        self.cfg = cfg
        
    # 功率归一化
    def _norm(self, x):
        pwr = np.mean(np.abs(x) ** 2, axis=1, keepdims=True)
        return x / np.sqrt(pwr + 1e-12)
    
    # 衰落类型
    # 平坦瑞利信道
    def _flat_rayleigh(self, x):
        N = x.shape[0] # 获取输入信号的长度（或批次大小）
        h = (np.random.randn(N, 1) + 1j * np.random.randn(N, 1)) / np.sqrt(2) # 信道系数
        # 信号x经过平坦瑞利信道后输出为x * h
        return x * h
    
    # 平坦莱斯衰落信道
    def _flat_rician(self, x):
        N = x.shape[0]
        K = 10 ** (self.cfg['rician_k_db'] / 10)
        phi = np.random.uniform(0, 2 * np.pi, (N, 1))
        h_los = np.sqrt(K / (K+1)) * np.exp(1j * phi)
        h_nlos = np.sqrt(1 / (2 * (K + 1))) * (np.random.randn(N, 1) + 1j * np.random.randn(N, 1))
        return x * (h_los + h_nlos)
    
    def _multipath(self, x):
        """
        多径频率选择性衰落
        指数功率时延谱: PDP(k) = exp(-k / τ_rms)
        每条径独立 Rayleigh 衰落
        """
        N, L = x.shape
        n_taps = self.cfg['multipath_taps']
        tau_rms = self.cfg['multipath_tau_rms']

        pdp = np.exp(-np.arange(n_taps, dtype=np.float64) / tau_rms)
        pdp /= pdp.sum()

        h = np.zeros((N, n_taps), dtype=np.complex128)
        for k in range(n_taps):
            h[:, k] = np.sqrt(pdp[k] / 2) * (np.random.randn(N) + 1j * np.random.randn(N))

        y = np.zeros_like(x)
        for k in range(n_taps):
            if k == 0:
                y += h[:, 0:1] * x
            else:
                y[:, k] += h[:, k:k + 1] * x[:, :-k]
        return y
    
    def _apply_mixed(self, x):
        N = x.shape[0]
        w = self.cfg['mixed_weights']
        types, probs = list(w.keys()), list(w.values())
        assign = np.random.choice(types, size=N, p=probs)

        y = x.copy()
        for ch in types:
            m = (assign == ch)
            if not m.any():
                continue
            sub = x[m]
            if ch == 'flat_rayleigh':
                sub = self._flat_rayleigh(sub)
            elif ch == 'flat_rician':
                sub = self._flat_rician(sub)
            elif ch == 'multipath':
                sub = self._multipath(sub)
            y[m] = sub
        return y
    
    # 噪声
    def _awgn(self, x, snr_db):
        sigma2 = 10.0 ** (-snr_db / 10.0)
        noise = np.sqrt(sigma2 / 2) * (np.random.randn(*x.shape) + 1j * np.random.randn(*x.shape))
        return x + noise

    # 主接口
    def apply(self, signals, snr_db):
        x = self._norm(signals)
        ch = self.cfg['channel_type']

        if ch == 'mixed':
            x = self._apply_mixed(x)
        elif ch == 'flat_rayleigh':
            x = self._flat_rayleigh(x)
        elif ch == 'flat_rician':
            x = self._flat_rician(x)
        elif ch == 'multipath':
            x = self._multipath(x)

        return self._awgn(x, snr_db).astype(np.complex64)

# NumPy 切片详解（以多径卷积为例）

## 前提假设

设 $N=2,\; L=6,\; K=3$（`n_taps=3`）

```
x 的列索引:  [0, 1, 2, 3, 4, 5]    shape = (N, 6)
h 的列索引:  [0, 1, 2]              shape = (N, 3)
y 的列索引:  [0, 1, 2, 3, 4, 5]    shape = (N, 6)
```

---

## 一、基础切片对照表

| 切片语法 | 含义 | 选取的列索引 | 输出 shape | 关键点 |
|:---------|:-----|:------------|:-----------|:-------|
| `h[:, 0]` | 第 0 列 | `[0]` | `(N,)` → **一维** | ⚠️ 降维了 |
| `h[:, 0:1]` | 第 0 到 0 列 | `[0]` | `(N, 1)` → **二维** | ✅ 保持二维，可广播 |
| `h[:, k:k+1]` | 第 k 到 k 列 | `[k]` | `(N, 1)` → **二维** | ✅ 保持二维，可广播 |
| `x[:, :-k]` | 从头到倒数第 k 列（不含） | 去掉最后 k 列 | `(N, L-k)` | 截掉尾部 |
| `y[:, k:]` | 从第 k 列到末尾 | 去掉前 k 列 | `(N, L-k)` | 截掉头部 |
| `x[:, :]` | 全部列 | `[0,1,...,L-1]` | `(N, L)` | 等同于 `x` |

---

## 二、每轮循环的具体切片

### **k=0 时**

```python
y += h[:, 0:1] * x
```

| 变量 | 切片 | 选取列 | shape |
|:-----|:-----|:------|:------|
| `h[:, 0:1]` | 第 0 列（保持二维） | `[0]` | `(N, 1)` |
| `x` | 全部列 | `[0,1,2,3,4,5]` | `(N, 6)` |
| **广播乘法结果** | — | — | `(N, 6)` |
| `y` | 全部列 | `[0,1,2,3,4,5]` | `(N, 6)` |

$$y[l] \mathrel{+}= h[0] \cdot x[l], \quad l = 0,1,2,3,4,5$$

```
h[:,0:1]:   [ h0 ]          (N,1)
              ↓ 广播到6列
x:          [ x0  x1  x2  x3  x4  x5 ]   (N,6)
结果:       [h0·x0  h0·x1  h0·x2  h0·x3  h0·x4  h0·x5]
y 更新位置:  [ ✓     ✓     ✓     ✓     ✓     ✓  ]
```

---

### **k=1 时**

```python
y[:, 1:] += h[:, 1:2] * x[:, :-1]
```

| 变量 | 切片 | 选取列 | shape |
|:-----|:-----|:------|:------|
| `h[:, 1:2]` | 第 1 列（保持二维） | `[1]` | `(N, 1)` |
| `x[:, :-1]` | 去掉最后 1 列 | `[0,1,2,3,4]` | `(N, 5)` |
| **广播乘法结果** | — | — | `(N, 5)` |
| `y[:, 1:]` | 去掉前 1 列 | `[1,2,3,4,5]` | `(N, 5)` |

$$y[l] \mathrel{+}= h[1] \cdot x[l-1], \quad l = 1,2,3,4,5$$

```
h[:,1:2]:   [ h1 ]                        (N,1)
              ↓ 广播到5列
x[:,:-1]:   [ x0   x1   x2   x3   x4 ]   (N,5)  ← 去掉 x5
结果:       [h1·x0  h1·x1  h1·x2  h1·x3  h1·x4]
y 更新位置:  [  ×     ✓     ✓     ✓     ✓     ✓ ]  ← y[:,1:]
              ↑
            y[0]不动
```

---

### **k=2 时**

```python
y[:, 2:] += h[:, 2:3] * x[:, :-2]
```

| 变量 | 切片 | 选取列 | shape |
|:-----|:-----|:------|:------|
| `h[:, 2:3]` | 第 2 列（保持二维） | `[2]` | `(N, 1)` |
| `x[:, :-2]` | 去掉最后 2 列 | `[0,1,2,3]` | `(N, 4)` |
| **广播乘法结果** | — | — | `(N, 4)` |
| `y[:, 2:]` | 去掉前 2 列 | `[2,3,4,5]` | `(N, 4)` |

$$y[l] \mathrel{+}= h[2] \cdot x[l-2], \quad l = 2,3,4,5$$

```
h[:,2:3]:   [ h2 ]                    (N,1)
              ↓ 广播到4列
x[:,:-2]:   [ x0   x1   x2   x3 ]    (N,4)  ← 去掉 x4, x5
结果:       [h2·x0  h2·x1  h2·x2  h2·x3]
y 更新位置:  [  ×     ×     ✓     ✓     ✓     ✓ ]  ← y[:,2:]
```

---

## 三、全局视图：y 的每个位置如何累加

| y 的列 | k=0 贡献 | k=1 贡献 | k=2 贡献 | **求和公式** |
|:------:|:--------:|:--------:|:--------:|:------------|
| `y[0]` | $h_0 \cdot x_0$ | — | — | $h_0 x_0$ |
| `y[1]` | $h_0 \cdot x_1$ | $h_1 \cdot x_0$ | — | $h_0 x_1 + h_1 x_0$ |
| `y[2]` | $h_0 \cdot x_2$ | $h_1 \cdot x_1$ | $h_2 \cdot x_0$ | $h_0 x_2 + h_1 x_1 + h_2 x_0$ |
| `y[3]` | $h_0 \cdot x_3$ | $h_1 \cdot x_2$ | $h_2 \cdot x_1$ | $h_0 x_3 + h_1 x_2 + h_2 x_1$ |
| `y[4]` | $h_0 \cdot x_4$ | $h_1 \cdot x_3$ | $h_2 \cdot x_2$ | $h_0 x_4 + h_1 x_3 + h_2 x_2$ |
| `y[5]` | $h_0 \cdot x_5$ | $h_1 \cdot x_4$ | $h_2 \cdot x_3$ | $h_0 x_5 + h_1 x_4 + h_2 x_3$ |

> 每行恰好就是卷积公式 $y[l] = \sum_{k} h[k] \cdot x[l-k]$

---

## 四、核心记忆口诀

```
┌──────────────────────────────────────────────────┐
│  h[:, k:k+1]  →  取第k条径的系数，保持二维(N,1)   │
│  x[:, :-k]    →  x 砍掉尾部 k 列 = x[l-k] 对齐  │
│  y[:, k:]     →  y 砍掉头部 k 列 = 从第k个位置写入 │
│                                                    │
│  两者 shape 完全匹配: 都是 (N, L-k)               │
│  h[:, k:k+1] 通过广播 (N,1) → (N, L-k)           │
└──────────────────────────────────────────────────┘
```

**本质：** 第 $k$ 条径让信号延迟了 $k$ 个样本，所以 `x` 要从右边砍掉 $k$ 列（未来的数据还没到），`y` 从左边跳过 $k$ 列（前 $k$ 个位置没有第 $k$ 径的贡献），两者刚好**对齐**。

# 多径频率选择性衰落信道 — 数学公式

---

## 1. 信道模型总览

接收信号是发送信号与信道冲激响应的**卷积**：

$$\boxed{y_n[l] = \sum_{k=0}^{K-1} h_n[k] \cdot x_n[l-k]}$$

其中：
- $n = 1, \ldots, N$ 为第 $n$ 个独立信道实现（批次索引）
- $l = 0, 1, \ldots, L-1$ 为时间/样本索引
- $K$ = `n_taps`，多径抽头数

---

## 2. 功率时延谱（Power Delay Profile, PDP）

采用**指数衰减**模型：

$$\widetilde{P}(k) = e^{-k / \tau_{\text{rms}}}, \quad k = 0, 1, \ldots, K-1$$

其中 $\tau_{\text{rms}}$ 是**均方根时延扩展**（`multipath_tau_rms`）。

### 归一化

为保证信道**总功率为 1**（不改变平均信号功率）：

$$P(k) = \frac{\widetilde{P}(k)}{\displaystyle\sum_{j=0}^{K-1} \widetilde{P}(j)} = \frac{e^{-k/\tau_{\text{rms}}}}{\displaystyle\sum_{j=0}^{K-1} e^{-j/\tau_{\text{rms}}}}$$

满足：

$$\sum_{k=0}^{K-1} P(k) = 1$$

---

## 3. 各径信道系数

每条径独立服从**瑞利衰落**，功率由 PDP 决定：

$$h_n[k] = \sqrt{\frac{P(k)}{2}} \left( g_{\text{re}}^{(n,k)} + j \cdot g_{\text{im}}^{(n,k)} \right)$$

其中：

$$g_{\text{re}}^{(n,k)} \sim \mathcal{N}(0,1), \quad g_{\text{im}}^{(n,k)} \sim \mathcal{N}(0,1)$$

各径、各实现之间**相互独立**。

因此：

$$\boxed{h_n[k] \sim \mathcal{CN}\big(0,\; P(k)\big)}$$

| 统计量 | 值 |
|:---:|:---:|
| $\mathbb{E}[h_n[k]]$ | $0$ |
| $\mathbb{E}\big[\|h_n[k]\|^2\big]$ | $P(k)$ |
| $\|h_n[k]\|$ 的分布 | $\text{Rayleigh}\!\left(\sqrt{P(k)/2}\right)$ |
| $\sum_k \mathbb{E}\big[\|h_n[k]\|^2\big]$ | $1$（功率归一化） |

---

## 4. 卷积输出（Tapped Delay Line）

代码实现的是**因果 FIR 滤波器**：

$$y_n[l] = \sum_{k=0}^{K-1} h_n[k] \cdot x_n[l - k]$$

展开写：

$$y_n[l] = \underbrace{h_n[0] \cdot x_n[l]}_{\text{直射径/主径}} + \underbrace{h_n[1] \cdot x_n[l-1]}_{\text{第1条多径}} + \cdots + \underbrace{h_n[K\!-\!1] \cdot x_n[l\!-\!K\!+\!1]}_{\text{第}K\!-\!1\text{条多径}}$$

> 其中 $x_n[l-k] = 0$ 当 $l - k < 0$（因果边界条件）。

---

## 5. 频域解释

信道的**频率响应**为冲激响应的 DFT：

$$H_n(f) = \sum_{k=0}^{K-1} h_n[k] \, e^{-j 2\pi f k}$$

- 当 $K = 1$（单径）：$H_n(f) = h_n[0]$，**平坦衰落**
- 当 $K > 1$（多径）：$H_n(f)$ 随频率变化，**频率选择性衰落**

---

## 6. 完整数学总结

$$\boxed{
\begin{aligned}
&\textbf{PDP:} \quad P(k) = \frac{e^{-k/\tau_{\text{rms}}}}{\sum_{j=0}^{K-1} e^{-j/\tau_{\text{rms}}}}, \quad k=0,\ldots,K\!-\!1 \\[8pt]
&\textbf{信道系数:} \quad h_n[k] \sim \mathcal{CN}\big(0, P(k)\big), \quad \text{各径独立} \\[8pt]
&\textbf{接收信号:} \quad y_n[l] = \sum_{k=0}^{K-1} h_n[k] \cdot x_n[l-k] = (h_n * x_n)[l]
\end{aligned}
}$$

---

## 图示：指数 PDP 形状

```
P(k)
 │
 █  ← 主径（最强）
 │█
 │ █
 │  █
 │   █▄
 │     ▄▄▄___
 └──────────────→ k (时延抽头)
   0  1  2  3  ...  K-1
```

> **物理意义**：电磁波经不同路径到达接收端，延迟越大的路径功率越弱（指数衰减），每条路径经历独立的瑞利衰落，多径叠加导致信号在频域上出现选择性衰减（频率选择性衰落），造成**符号间干扰（ISI）**。

# 平坦瑞利衰落信道
### 第一步：确定系统模型（映射 `return x * h`）
**1. 理论原理**
在无线通信基带等效模型中，如果信号经历的是**平坦衰落（Flat Fading）**，意味着信号的带宽小于信道的相干带宽。
*   **物理结论**：此时信道对信号的影响仅仅是幅度的衰减和相位的旋转，**没有多径时延造成的符号间干扰（ISI）**。
*   **数学公式**：接收信号 $y$ 等于发送信号 $x$ 直接乘以一个复数信道系数 $h$（忽略噪声）：
    $$ y = h \cdot x $$

**2. 映射为代码**
因为是简单的相乘关系，不涉及卷积（np.convolve），所以在代码中直接映射为逐元素相乘（Hadamard积）：
```python
return x * h
```

---

### 第二步：确定信道系数 $h$ 的数学属性
**1. 理论原理**
在没有直射视距路径（Non-Line-of-Sight）的复杂城市环境中，根据**中心极限定理**，大量独立散射路径叠加后，接收端信道系数的实部和虚部均服从高斯分布，这被称为**瑞利衰落信道**。
*   **数学公式**：信道系数 $h$ 是一个**零均值复高斯随机变量**（Circularly Symmetric Complex Gaussian），记为：
    $$ h \sim \mathcal{CN}(0, \sigma^2) $$
*   即 $h = X + jY$，其中实部 $X$ 和虚部 $Y$ 是独立的高斯随机变量，且均值为 0。

**2. 确定方差（功率归一化）**
为了在仿真中不改变发射信号的平均功率，我们规定信道的平均功率（即方差）为 1：
$$ \mathbb{E}[|h|^2] = 1 $$
因为 $|h|^2 = X^2 + Y^2$，且 $X$ 和 $Y$ 独立同分布，所以：
$$ \mathbb{E}[X^2] + \mathbb{E}[Y^2] = 1 $$
这就要求实部和虚部的方差各占一半：
$$ \text{Var}(X) = 0.5, \quad \text{Var}(Y) = 0.5 $$
即 $X \sim \mathcal{N}(0, 0.5)$， $Y \sim \mathcal{N}(0, 0.5)$。

---

### 第三步：将数学分布映射为 NumPy 代码（映射 `h = ...`）
**1. 理论到工具的转换**
NumPy 提供的标准正态分布生成函数是 `np.random.randn()`，它生成的是均值为 0、**方差为 1** 的标准正态分布 $U \sim \mathcal{N}(0, 1)$。

根据概率论的性质，如果我们想要得到方差为 $0.5$ 的高斯变量，我们需要对标准正态分布进行缩放：
因为 $\text{Var}(c \cdot U) = c^2 \cdot \text{Var}(U)$，
令 $c^2 = 0.5$，则 $c = \frac{1}{\sqrt{2}}$。

所以，我们需要将 `randn()` 生成的数值**除以 $\sqrt{2}$**，才能得到方差为 0.5 的实部和虚部：
$$ X = \frac{\mathcal{N}(0, 1)}{\sqrt{2}}, \quad Y = \frac{\mathcal{N}(0, 1)}{\sqrt{2}} $$

**2. 组装成最终公式**
$$ h = X + jY = \frac{\mathcal{N}(0, 1) + j \cdot \mathcal{N}(0, 1)}{\sqrt{2}} $$

**3. 映射为代码**
根据上述公式，要生成 $N$ 个这样的复数样本（保持和输入信号 $x$ 的长度一致，`N = x.shape[0]`），直接翻译成 NumPy 代码就是：
*   $\mathcal{N}(0, 1)$ 对应 `np.random.randn(N, 1)`
*   $j$ 对应 Python 的虚数单位 `1j`
*   $\sqrt{2}$ 对应 `np.sqrt(2)`

最终拼装出的代码就是：
```python
h = (np.random.randn(N, 1) + 1j * np.random.randn(N, 1)) / np.sqrt(2)
```

### 总结
基于以下三个核心原理：
1.  **平坦衰落原理** $\rightarrow$ 映射为直接相乘 (`x * h`)。
2.  **瑞利信道包络原理** $\rightarrow$ 映射为实部和虚部均服从高斯分布，组成复高斯变量 (`randn + 1j * randn`)。
3.  **信道功率归一化原理** $\rightarrow$ 映射为将方差从 1 缩放到 0.5，以此保持信号功率不被异常放大 (`/ np.sqrt(2)`)。

**平坦莱斯衰落信道（Flat Rician Fading Channel）**。
“莱斯（Rician）信道”的物理场景是：**发射机和接收机之间存在一条直射的视距路径（Line-of-Sight, LoS），同时还存在许多散射的非视距路径（Non-Line-of-Sight, NLoS）。**

### 第一步：莱斯因子 $K$ 转换为线性值
```python
K = 10 ** (self.cfg['rician_k_db'] / 10)
```
**数学公式：**
$$ K = 10^{\frac{K_{dB}}{10}} $$
**原理解释：**
莱斯因子 $K$ 是莱斯信道最重要的参数，定义为**“直射路径功率”与“所有散射路径总功率”的比值**。
在配置文件中，它通常以分贝（dB）为单位配置（`rician_k_db`），这行代码利用对数运算的逆运算，将其转换为了普通的**线性比值** $K$。

---

### 第二步：生成直射路径（LoS）分量 $h_{LoS}$
```python
phi = np.random.uniform(0, 2 * np.pi, (N, 1))
h_los = np.sqrt(K / (K+1)) * np.exp(1j * phi)
```
**数学公式：**
对于第 $k$ 个采样点，直射路径分量可以表示为：
$$ h_{LoS, k} = \sqrt{\frac{K}{K+1}} \cdot e^{j \phi_k} $$
其中 $\phi_k \sim \mathcal{U}[0, 2\pi)$
**原理解释：**
*   **相位 $e^{j \phi_k}$**：虽然是直射路径，但由于收发端距离的微小变化，直射信号到达接收端时的**相位是随机的**。代码用 `np.random.uniform(0, 2*pi)` 生成了均匀分布的随机相位 $\phi$。
*   **幅度 $\sqrt{\frac{K}{K+1}}$**：这是一个固定的常数，代表直射路径的幅度衰减。为什么是这个式子，我们在最后会解释（为了功率归一化）。

---

### 第三步：生成散射路径（NLoS）分量 $h_{NLoS}$
```python
h_nlos = np.sqrt(1 / (2 * (K + 1))) * (np.random.randn(N, 1) + 1j * np.random.randn(N, 1))
```
**数学公式：**
$$ h_{NLoS, k} = \sqrt{\frac{1}{K+1}} \cdot \underbrace{\left( \frac{\mathcal{N}(0, 1) + j \cdot \mathcal{N}(0, 1)}{\sqrt{2}} \right)}_{\text{标准的瑞利衰落}} $$
结合代码里的根号合并，可以写成：
$$ h_{NLoS, k} = \sqrt{\frac{1}{2(K+1)}} \cdot (\mathcal{N}(0, 1) + j \cdot \mathcal{N}(0, 1)) $$
**原理解释：**
你会发现，括号里的 `(randn + 1j * randn)` 就是我们之前讲的**瑞利衰落**（代表纯散射）。
莱斯信道里的 NLoS 部分本质上就是一个被缩小了幅度的瑞利信道。它的幅度缩放因子是 $\sqrt{\frac{1}{K+1}}$。

---

### 第四步：总信道与输出信号 $y$
```python
return x * (h_los + h_nlos)
```
**数学公式：**
1. **信道叠加原理**：总信道系数 $h_k$ 等于直射路径和散射路径叠加：
   $$ h_k = h_{LoS, k} + h_{NLoS, k} $$
2. **平坦衰落相乘**：输出信号 $y_k$ 等于输入信号 $x_k$ 乘以总信道系数：
   $$ y_k = x_k \cdot h_k = x_k \cdot (h_{LoS, k} + h_{NLoS, k}) $$

---

### 核心推导：为什么系数是 $\frac{K}{K+1}$ 和 $\frac{1}{K+1}$？

这是这段代码在数学映射上最精妙的地方，也是为了实现**“功率归一化（信道平均功率为 1）”**。

设总信道功率为 $\Omega = \mathbb{E}[|h|^2] = 1$。
根据定义，$K = \frac{\text{直射功率}}{\text{散射功率}}$，且 $\text{直射功率} + \text{散射功率} = 1$。
解这个方程组：
1. **直射功率 (LoS Power)** $= \frac{K}{K+1}$
   所以直射分量 $h_{LoS}$ 的**幅度**必须是它的平方根： $\sqrt{\frac{K}{K+1}}$。
2. **散射功率 (NLoS Power)** $= \frac{1}{K+1}$
   所以散射分量 $h_{NLoS}$ 的**方差**必须是 $\frac{1}{K+1}$。
   因为 `randn + 1j * randn` 的方差是 2，所以要给它乘上 $\sqrt{\frac{1}{2(K+1)}}$，平方后刚好等于 $\frac{1}{K+1}$。

### 总结
信号 $x$ 经过了平坦莱斯信道，输出等于 $x$ 乘以信道系数 $h$；而 $h$ 是由**确定幅度的直射路径（占比 $\frac{K}{K+1}$）**和**高斯分布的随机散射路径（占比 $\frac{1}{K+1}$）**共同叠加而成的。

## 代码流程图

```
apply()
  │
  ├─① self._norm(x)            ← 功率归一化
  │
  ├─② self._apply_mixed(x)     ← 衰落处理（不含噪声）
  │
  └─③ self._awgn(x, snr_db)    ← 所有信号最后都加 AWGN
                                    ↑
                                    无论走哪个分支，都会执行
```
---

$$\boxed{y = \underbrace{f_{\text{fading}}(x)}_{\text{混合信道按比例选一个}} + \underbrace{n}_{\text{所有样本都加 AWGN}}}$$

- **`mixed_weights` 中的 `'awgn': 0.30`** = 30% 的样本**只经历纯 AWGN**（不加任何衰落）
- **其余 70%** 的样本 = **衰落 + AWGN**（先衰落，再加噪声）

# `_apply_mixed` 循环逻辑详解

## 先看循环之前发生了什么

```python
N = x.shape[0]                                          # 比如 N=10
w = self.cfg['mixed_weights']
types  = ['awgn', 'flat_rayleigh', 'flat_rician', 'multipath']
probs  = [0.30,   0.20,            0.15,           0.35]

assign = np.random.choice(types, size=N, p=probs)       # 每个样本抽签
y = x.copy()                                            # 先复制一份
```

**`assign` 就是一个长度为 N 的"分配表"，每个样本被随机分到一种信道。**

---

## 用 N=10 的具体例子

假设随机抽签结果：

```
样本索引:   0          1              2           3          4
assign:  "multipath" "awgn"    "flat_rayleigh" "multipath" "awgn"

样本索引:   5              6          7              8           9
assign:  "flat_rician" "multipath" "awgn"    "flat_rayleigh" "multipath"
```

---

## 循环逐轮执行

### **第 1 轮：`ch = 'awgn'`**

```python
m = (assign == 'awgn')
# m = [F, T, F, F, T, F, F, T, F, F]
#      0  1  2  3  4  5  6  7  8  9
```

```python
sub = x[m]          # 取出样本 1, 4, 7
```

```python
# 没有任何 if 分支匹配 'awgn'，所以 sub 不变
if ch == 'flat_rayleigh':   # ✗
elif ch == 'flat_rician':   # ✗
elif ch == 'multipath':     # ✗
```

```python
y[m] = sub          # 原样写回 → 样本 1,4,7 无衰落
```

```
y:  [原  原  原  原  原  原  原  原  原  原]
         ↑           ↑           ↑
       1号不变     4号不变     7号不变
```

---

### **第 2 轮：`ch = 'flat_rayleigh'`**

```python
m = (assign == 'flat_rayleigh')
# m = [F, F, T, F, F, F, F, F, T, F]
#      0  1  2  3  4  5  6  7  8  9
```

```python
sub = x[m]                       # 取出样本 2, 8
sub = self._flat_rayleigh(sub)   # 施加瑞利衰落 ✓
y[m] = sub                       # 写回到 y 的第 2, 8 位
```

```
y:  [原  原  瑞  原  原  原  原  原  瑞  原]
              ↑                       ↑
           2号瑞利                  8号瑞利
```

---

### **第 3 轮：`ch = 'flat_rician'`**

```python
m = (assign == 'flat_rician')
# m = [F, F, F, F, F, T, F, F, F, F]
#      0  1  2  3  4  5  6  7  8  9
```

```python
sub = x[m]                     # 取出样本 5
sub = self._flat_rician(sub)   # 施加莱斯衰落 ✓
y[m] = sub                     # 写回到 y 的第 5 位
```

```
y:  [原  原  瑞  原  原  莱  原  原  瑞  原]
                        ↑
                     5号莱斯
```

---

### **第 4 轮：`ch = 'multipath'`**

```python
m = (assign == 'multipath')
# m = [T, F, F, T, F, F, T, F, F, T]
#      0  1  2  3  4  5  6  7  8  9
```

```python
sub = x[m]                    # 取出样本 0, 3, 6, 9
sub = self._multipath(sub)    # 施加多径衰落 ✓
y[m] = sub                    # 写回到 y 的第 0,3,6,9 位
```

```
y:  [多  原  瑞  多  原  莱  多  原  瑞  多]
     ↑       ↑                  ↑           ↑
  0号多径  3号多径           6号多径     9号多径
```

---

## 最终结果一览

| 样本 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|:----:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| assign | 多径 | awgn | 瑞利 | 多径 | awgn | 莱斯 | 多径 | awgn | 瑞利 | 多径 |
| y 经历的衰落 | ✅多径 | ❌无 | ✅瑞利 | ✅多径 | ❌无 | ✅莱斯 | ✅多径 | ❌无 | ✅瑞利 | ✅多径 |

> 之后 **所有 10 个样本** 都会再经过 `_awgn()` 加噪声。

---

## 流程总结

```
┌─────────────────────────────────────────────────┐
│  Step 1: 给每个样本「抽签」决定信道类型           │
│          assign = random.choice(types, p=probs)  │
│                                                  │
│  Step 2: 按类型「分组处理」                       │
│                                                  │
│     for 每种信道类型 ch:                          │
│         ① m = (assign == ch)   ← 布尔掩码        │
│         ② sub = x[m]          ← 取出该组样本      │
│         ③ sub = 衰落(sub)     ← 只处理这一组      │
│         ④ y[m] = sub          ← 写回原位置        │
│                                                  │
│  为什么不一个一个处理？                            │
│  → 向量化批处理，速度快几十倍！                    │
└─────────────────────────────────────────────────┘
```

**核心思路就是「分组 → 批处理 → 写回」**，布尔掩码 `m` 负责筛选，`y[m] = sub` 负责精准归位。

# `if not m.any(): continue` 详解

## 逐步拆解

| 部分 | 含义 |
|:-----|:-----|
| `m` | 布尔数组，如 `[False, True, False, False, True, ...]` |
| `m.any()` | m 中**是否存在至少一个 True** |
| `not m.any()` | m 中**一个 True 都没有**（全是 False） |
| `continue` | **跳过本轮循环**，直接进入下一个 `ch` |

---

## 举个例子

假设 N=6，抽签结果恰好没有人被分到 `'flat_rician'`：

```
assign = ["awgn", "multipath", "awgn", "flat_rayleigh", "multipath", "awgn"]
```

当循环到 `ch = 'flat_rician'` 时：

```python
m = (assign == 'flat_rician')
# m = [False, False, False, False, False, False]   ← 全是 False

m.any()        # → False（没有任何一个 True）
not m.any()    # → True

# 进入 if 分支 → continue → 跳过，不执行后面的衰落处理
```

---

## 为什么需要这个判断？

如果不加这个判断，当 `m` 全为 False 时：

```python
sub = x[m]    # → shape = (0, L)，空数组
sub = self._flat_rician(sub)   # 对空数组做运算 → 可能出错或浪费时间
y[m] = sub
```

加了之后：

```
没人被分到这种信道 → 直接跳过 → 省时间、避免空数组问题
```

---

## 一句话总结

> **"如果没有任何样本被分配到当前信道类型，就跳过，不做无用功。"**

In [21]:
# 6.RF损伤
class RFImpairments:
    """
    施加顺序（符合实际硬件链路）:
      TX: PA 非线性
      RX: CFO → 相位噪声 → IQ 不平衡 → DC 偏移 → 采样时偏
    """
    def __init__(self, cfg=CONFIG):
        self.cfg = cfg

    # TX侧
    def apply_tx(self, x):
        if self.cfg['pa_enable']:
            x = self._pa_rapp(x)
        return x
    
    def _pa_rapp(self, x):
        """
        Rapp SSPA 模型 (仅 AM/AM，无 AM/PM)
        G(r) = r / [1 + (r/A_sat)^{2p}]^{1/(2p)}
        """
        p = self.cfg['pa_p']
        ibo = self.cfg['pa_ibo_db']
        pwr = np.mean(np.abs(x) ** 2, axis=-1, keepdims=True)
        A_sat = np.sqrt(pwr  * 10 ** (ibo / 10))

        r = np.abs(x)
        theta = np.angle(x)
        r_out = r / (1 + (r / (A_sat + 1e-12)) ** (2 * p)) ** (1 / (2 * p))
        return r_out * np.exp(1j * theta)
    
    # RX侧
    def apply_rx(self, x):
        if not self.cfg['impairments_enable']:
            return x
        x = self._cfo(x)
        x = self._phase_noise(x)
        x = self._iq_imbalance(x)
        x = self._dc_offset(x)
        x = self._timing_offset(x)
        return x
    
    def _cfo(self, x):
        """载波频偏: x · exp(j·(2π·Δf·n + φ₀))"""
        N, L = x.shape
        max_cfo = self.cfg['cfo_max_norm']
        if max_cfo <= 0:
            return x
        df = np.random.uniform(-max_cfo, max_cfo, (N, 1))
        phi = np.random.uniform(0, 2 * np.pi, (N, 1))
        n = np.arange(L).reshape(1, L)
        return x * np.exp(1j * (2 * np.pi * df * n + phi))
    
    def _phase_noise(self, x):
        """
        振荡器相位噪声 (Wiener 过程)
        φ[n] = φ[n-1] + N(0, σ²)
        """
        N, L = x.shape
        sigma = self.cfg['phase_noise_std']
        if sigma <= 0:
            return x
        increments = sigma * np.random.randn(N, L)
        pn = np.cumsum(increments, axis=1)
        return x * np.exp(1j * pn)
    
    def _iq_imbalance(self, x):
        """
        IQ 不平衡 (直接变频接收机模型)

        I_out = (1 + ε/2) · I
        Q_out = (1 − ε/2) · [Q·cos(Δφ) + I·sin(Δφ)]

        ε: 增益不平衡 (线性)
        Δφ: 相位不平衡 (rad)
        """
        N, L = x.shape
        g_max = self.cfg['iq_gain_imb_max_db']
        p_max = self.cfg['iq_phase_imb_max_deg']
        if g_max <= 0 and p_max <= 0:
            return x
        # eps → ε
        eps = 10 ** (np.random.uniform(-g_max, g_max, (N, 1)) / 20) -1 # dB → 线性偏差
        # dphi → Δφ
        dphi = np.random.uniform(-p_max, p_max, (N, 1)) * np.pi / 180
        
        I, Q = x.real, x.imag
        I_out = (1 + eps) * I
        Q_out = (1 - eps) * [Q * np.cos(dphi) + I * np.sin(dphi)]
        return I_out + 1j * Q_out
    
    def _dc_offset(self, x):
        """ADC 直流泄漏"""
        N, L = x.shape
        mx = self.cfg['dc_offset_max']
        if mx <= 0:
            return x
        dc = np.random.uniform(-mx, mx, (N, 1)) + 1j * np.random.uniform(-mx, mx, (N, 1))
        return x + dc
    
    def _timing_offset(self, x):
        """分数采样时偏 (线性插值)"""
        N, L = x.shape
        mx = self.cfg['timing_offset_max']
        if mx <= 0:
            return x
        offsets = np.random.uniform(-mx, mx, N)
        t = np.arange(L, dtype=np.float64)
        y = np.empty_like(x)
        for i in range(N):
            t_new = np.clip(t + offsets[i], 0, L - 1)
            y[i] = np.interp(t_new, t, x[i].real) + 1j * np.interp(t_new, t, x[i].imag)
        return y

# RF 损伤对调制信号的影响详解

---

## 一、各损伤的独立影响

### 1. PA 非线性（TX 侧）

$$G(r) = \frac{r}{\left[1 + \left(\frac{r}{A_{\text{sat}}}\right)^{2p}\right]^{\frac{1}{2p}}}$$

| 影响 | 说明 |
|:-----|:-----|
| **幅度压缩** | 大信号被压缩，小信号不受影响 |
| **星座图畸变** | 外圈符号向内"挤压"，内圈基本不动 |
| **频谱再生** | 产生带外杂散（谱扩展） |
| **EVM 恶化** | 高阶调制（64QAM、256QAM）最敏感 |

**星座图效果：**
```
理想 16QAM              PA 后 16QAM
·   ·   ·   ·           ·   ·   ·   ·
                         
·   ·   ·   ·           ·  ·   ·  ·      ← 内圈不变
                         
·   ·   ·   ·           ·  ·   ·  ·      
                         
·   ·   ·   ·           ·   ·   ·   ·    ← 外圈被压缩向内
```

> **受影响最大的调制**：高阶 QAM（256QAM > 128QAM > 64QAM），因为外圈点多、动态范围大。恒包络调制（如 MSK、FSK）基本不受影响。

---

### 2. 载波频偏 CFO

$$y[n] = x[n] \cdot e^{j(2\pi \Delta f \cdot n + \phi_0)}$$

| 影响 | 说明 |
|:-----|:-----|
| **星座旋转** | 整个星座图随时间持续旋转 |
| **旋转速度** | 与 $\Delta f$ 成正比 |
| **符号间相位差** | 相邻符号有固定相位增量 $2\pi\Delta f$ |

**星座图效果：**
```
理想 QPSK              CFO 后 QPSK

    ·                        ·  
                           ↗ 
·       ·              旋转成一个圆环
                           ↘
    ·                        ·
```

> **所有调制都受影响**，但差分调制（如 Pi/4-DQPSK）对 CFO 有天然抗性。

---

### 3. 相位噪声（Wiener 过程）

$$\phi[n] = \phi[n-1] + \mathcal{N}(0, \sigma^2)$$
$$y[n] = x[n] \cdot e^{j\phi[n]}$$

| 影响 | 说明 |
|:-----|:-----|
| **随机旋转** | 星座点围绕原点做随机角度抖动 |
| **"扩散"效果** | 点变成弧形扇区 |
| **累积性** | Wiener 过程方差随时间增长 $\text{Var}[\phi[n]] = n\sigma^2$ |

**星座图效果：**
```
理想 8PSK               相位噪声后 8PSK

    ·                        ···
  ·   ·                    ··   ··    ← 每个点沿圆周"抹开"
·       ·                ··       ··
  ·   ·                    ··   ··
    ·                        ···
```

> **PSK 类调制最敏感**（如 8PSK），因为全靠相位区分符号。QAM 也受影响但同时有幅度信息辅助。

---

### 4. IQ 不平衡

$$I_{\text{out}} = \left(1 + \frac{\varepsilon}{2}\right) I$$
$$Q_{\text{out}} = \left(1 - \frac{\varepsilon}{2}\right) \left[Q\cos\Delta\phi + I\sin\Delta\phi\right]$$

| 影响 | 说明 |
|:-----|:-----|
| **增益不平衡 $\varepsilon$** | 星座图沿 I 或 Q 轴拉伸/压缩 → 圆变椭圆 |
| **相位不平衡 $\Delta\phi$** | I、Q 轴不正交 → 星座图倾斜剪切 |
| **镜像干扰** | 产生镜像频率分量 |

**星座图效果：**
```
理想 QPSK                IQ不平衡后 QPSK

Q                        Q
↑   ·       ·            ↑   ·       ·
|                        |  ↗         ↗   ← 轴不正交
|                        | /         /        + 轴不等长
+-------→ I             +-------→ I
    ·       ·                ·       ·
```

等效于对信号施加一个**镜像干扰**：

$$y = \alpha \cdot x + \beta \cdot x^*$$

其中 $\alpha \approx 1$，$\beta$ 与不平衡程度成正比。

> **对称星座（如 QAM）** 受影响最大，因为镜像点可能落在其他符号位置上。

---

### 5. DC 偏移

$$y[n] = x[n] + (d_I + j \cdot d_Q)$$

| 影响 | 说明 |
|:-----|:-----|
| **整体平移** | 整个星座图偏离原点 |
| **对称性破坏** | 原本对称的判决区域变得不对称 |

**星座图效果：**
```
理想 QPSK                DC偏移后 QPSK

Q                        Q
↑   ·       ·            ↑
|                        |       ·       ·     ← 整体右上平移
|     ╋(原点)            |         ╋
+-------→ I             |   ·       ·
    ·       ·            +------------→ I
```

> **低阶调制（BPSK、QPSK）**：判决边界偏移，但间距大，容忍度高。**高阶 QAM**：内圈符号间距小，DC 偏移的相对影响更大。

---

### 6. 分数采样时偏

$$y[n] = x(nT_s + \delta T_s)$$

通过线性插值近似：
$$y[n] \approx (1-\delta) \cdot x[n] + \delta \cdot x[n+1]$$

| 影响 | 说明 |
|:-----|:-----|
| **符号间干扰 (ISI)** | 采样点偏离最佳眼图位置 |
| **星座点扩散** | 不是沿角度扩散，而是沿某方向"散开" |
| **眼图闭合** | 眼图张开度减小 |

**星座图效果：**
```
理想 QPSK                时偏后 QPSK

    ·       ·              ·:·     ·:·     ← 每个点变成小团
                           :::     :::
    ·       ·              ·:·     ·:·
```

> **高波特率、小滚降因子**的信号最敏感（RRC 滤波后的信号对采样时刻非常敏感）。

---

## 二、损伤叠加后的综合影响

### 叠加顺序（符合真实硬件链路）

```
x ──→ [PA非线性] ──→ [信道衰落+AWGN] ──→ [CFO] ──→ [相位噪声] ──→ [IQ不平衡] ──→ [DC偏移] ──→ [时偏] ──→ y
       TX侧              信道                              RX侧
```

### 综合效果

```
理想 16QAM                            所有损伤叠加后的 16QAM

·    ·    ·    ·                        .:·  :··  ·::  .:·
                                       ::·: :::  :·:: ·:::
·    ·    ·    ·           →           :·:: ··:· ::·: :·:·
                                       ·::: :·:· :::· ··::
·    ·    ·    ·                        ::·· :··: ·:·: ·:::
                                       :·:· ::·: :::· :·::
·    ·    ·    ·                        .:·: :·:  ·::  ·::·

每个点整齐分明                          每个点变成一团"云雾"
```

### 各损伤的叠加效应分解

| 损伤 | 对星座图的作用 | 几何比喻 |
|:-----|:-------------|:---------|
| PA 非线性 | 外圈向内压缩 | 压扁 |
| CFO | 持续旋转 | 转圈 |
| 相位噪声 | 随机角度抖动 | 抹开（弧形） |
| IQ 不平衡 | 拉伸 + 倾斜 | 变形 |
| DC 偏移 | 整体平移 | 搬家 |
| 时偏 | ISI 导致扩散 | 散开（团状） |
| **AWGN** | 各方向均匀扩散 | 高斯云 |

### 综合数学模型

$$\boxed{y[n] = \underbrace{\alpha}_{\text{IQ}} \cdot \Big[\underbrace{e^{j\phi[n]}}_{\text{相噪}} \cdot \underbrace{e^{j2\pi\Delta f \cdot n}}_{\text{CFO}} \cdot \underbrace{\sum_k h[k]}_{\text{信道}} \cdot \underbrace{G\big(x[n-k]\big)}_{\text{PA}} + \underbrace{w[n]}_{\text{AWGN}}\Big] + \underbrace{\beta \cdot (\cdot)^*}_{\text{IQ镜像}} + \underbrace{d}_{\text{DC}}}$$

---

## 三、各调制类型的敏感度总结

| 调制 | PA | CFO | 相噪 | IQ不平衡 | DC偏移 | 时偏 | 总体鲁棒性 |
|:-----|:--:|:---:|:----:|:-------:|:-----:|:----:|:--------:|
| BPSK | 🟢低 | 🟡中 | 🟢低 | 🟢低 | 🟢低 | 🟡中 | ⭐⭐⭐⭐⭐ |
| QPSK | 🟢低 | 🟡中 | 🟡中 | 🟡中 | 🟢低 | 🟡中 | ⭐⭐⭐⭐ |
| 8PSK | 🟢低 | 🟡中 | 🔴高 | 🟡中 | 🟡中 | 🟡中 | ⭐⭐⭐ |
| 16QAM | 🟡中 | 🟡中 | 🟡中 | 🟡中 | 🟡中 | 🟡中 | ⭐⭐⭐ |
| 64QAM | 🔴高 | 🔴高 | 🔴高 | 🔴高 | 🟡中 | 🔴高 | ⭐⭐ |
| 256QAM | 🔴高 | 🔴高 | 🔴高 | 🔴高 | 🔴高 | 🔴高 | ⭐ |
| MSK/FSK | 🟢低 | 🟢低 | 🟢低 | 🟡中 | 🟢低 | 🟡中 | ⭐⭐⭐⭐⭐ |
| Pi4DQPSK | 🟢低 | 🟢低 | 🟡中 | 🟡中 | 🟢低 | 🟡中 | ⭐⭐⭐⭐ |

> **规律：调制阶数越高 → 符号间距越小 → 对所有损伤越敏感。恒包络调制对 PA 天然免疫，差分调制对 CFO 有天然抗性。**

# CFO 函数中的运算符归属清单

## 代码逐行标注

```python
return x * np.exp(1j * (2 * np.pi * df * n + phi))
```

拆解为原子操作：

```python
a = 2 * np.pi          # ① ② 
b = df * n             # ③
c = a * b              # ④
d = c + phi            # ⑤
e = 1j * d             # ⑥
f = np.exp(e)          # ⑦
g = x * f              # ⑧
```

---

## 分类清单

### Python 内建运算符

| 运算符 | 出现位置 | 作用 | 说明 |
|:------:|:--------|:-----|:-----|
| `*` | ①③④⑥⑧ | 乘法 | Python 语法，但操作数是 ndarray 时**自动调用 NumPy 的逐元素乘法** |
| `+` | ⑤ | 加法 | 同上，ndarray 之间自动变成逐元素加法 |
| `**` | 本函数未出现，但 PA 中有 | 幂运算 | 同理，ndarray 时变成逐元素幂 |
| `-` | `(-max_cfo, ...)` | 取负 / 减法 | Python 语法 |
| `1j` | ⑥ | 虚数单位 | **Python 内建**，表示 $\sqrt{-1}$ |

### NumPy 函数 / 常量

| 符号 | 类型 | 作用 | 数学含义 |
|:-----|:-----|:-----|:---------|
| `np.pi` | 常量 | $\pi = 3.14159\ldots$ | 圆周率 |
| `np.exp()` | 函数 | 逐元素指数 $e^{(\cdot)}$ | $e^{j\theta}$ |
| `np.arange()` | 函数 | 生成整数序列 | $n = [0, 1, 2, \ldots, L\!-\!1]$ |
| `np.random.uniform()` | 函数 | 均匀分布随机数 | 生成 $\Delta f$, $\phi_0$ |

### NumPy 的 ndarray 方法

| 符号 | 类型 | 作用 |
|:-----|:-----|:-----|
| `.reshape(1, L)` | 方法 | 改变数组形状 (用于广播) |
| `.shape` | 属性 | 获取数组维度 |

---

## 关键概念：运算符重载

```
Python 的 * + - ** /  本身是通用运算符
         │
         ├─ 操作数是 int/float  → Python 原生运算
         │    2 * 3.14  →  6.28
         │
         └─ 操作数是 ndarray   → NumPy 接管（运算符重载）
              df * n    →  逐元素乘法 + 广播
```

所以同一个 `*` 号：

| 表达式 | 操作数类型 | 实际执行者 | 行为 |
|:-------|:----------|:----------|:-----|
| `2 * np.pi` | `int * float` | **Python** | 标量乘法 → `6.2831...` |
| `df * n` | `ndarray * ndarray` | **NumPy** | 逐元素乘法 + 广播 |
| `1j * d` | `complex * ndarray` | **NumPy** | 标量广播到数组，逐元素乘 |
| `x * f` | `ndarray * ndarray` | **NumPy** | 逐元素复数乘法 |

---

## 整行公式的数学对应

```python
x * np.exp(1j * (2 * np.pi * df * n + phi))
```

$$y_{i}[n] = x_{i}[n] \cdot \underbrace{e^{\,j\,(2\pi \,\Delta f_i \,\cdot\, n \;+\; \phi_{0,i})}}_{\text{np.exp}(\,\text{1j} \;\cdot\; (\underbrace{2}_{\text{int}} \;\underbrace{*}_{\text{Py}} \;\underbrace{\text{np.pi}}_{\pi} \;\underbrace{*}_{\text{NP}} \;\underbrace{\text{df}}_{\Delta f} \;\underbrace{*}_{\text{NP}} \;\underbrace{n}_{[0..L\!-\!1]} \;\underbrace{+}_{\text{NP}} \;\underbrace{\text{phi}}_{\phi_0})\,)}$$

---

## 一句话总结

> **`*` `+` `-` `**` 是 Python 语法糖，操作 ndarray 时自动委托给 NumPy 执行逐元素运算和广播。真正"只属于 NumPy"的是 `np.exp`、`np.pi`、`np.arange` 等函数和常量，以及 `1j` 是 Python 内建的虚数单位。**

In [ ]:
# 7. 调制信号生成器（全部矢量化）

# A. 通用线性调制: 星座 → 上采样 → RRC
def _gen_linear(const_name, n):
    syms = _rand_symbols(const_name, n)  # (N, NUM_SYM)
    up = np.zeros((n, NUM_SYM *SPS), dtype=np.complex128)
    up[:, ::SPS] = syms
    return batch_fft_convolve(up, _RRC)[:, :SIG_LEN]

# `_gen_linear` 函数详解

---

## 一、逐行解释

```python
def _gen_linear(const_name, n):
    syms = _rand_symbols(const_name, n)           # ①
    up = np.zeros((n, NUM_SYM * SPS), dtype=np.complex128)  # ②
    up[:, ::SPS] = syms                            # ③
    return batch_fft_convolve(up, _RRC)[:, :SIG_LEN]  # ④
```

### ① 随机选符号

```python
syms = _rand_symbols(const_name, n)    # shape = (n, 256)
```

从星座表中随机取 256 个符号，共 n 条信号。

### ② 创建上采样数组（全零）

```python
up = np.zeros((n, NUM_SYM * SPS), dtype=np.complex128)
#                   256 × 4 = 1024
# shape = (n, 1024)，全是 0
```

### ③ 插入符号（每隔 SPS=4 个位置放一个符号）

```python
up[:, ::SPS] = syms
#       ↑
#   步长为4：第0, 4, 8, 12, ... 个位置
```

具体效果（以 1 条信号为例）：

```
syms:  [s₀,     s₁,     s₂,     s₃,    ...]    256个符号

up:    [s₀, 0, 0, 0, s₁, 0, 0, 0, s₂, 0, 0, 0, s₃, 0, 0, 0, ...]
        ↑           ↑           ↑           ↑
       [0]         [4]         [8]        [12]     1024个采样点

        ← SPS=4 →
```

这就是**上采样（Upsampling）**：符号之间插入 3 个零。

### ④ RRC 脉冲成形 + 截断

```python
batch_fft_convolve(up, _RRC)[:, :SIG_LEN]
#       ↑                        ↑
#   FFT快速卷积               截取前1024个点
```

卷积会让信号变长（长度 = 1024 + 101 - 1 = 1124），所以截取前 `SIG_LEN=1024` 个点。

---

## 完整流程图

```
    ① 随机符号              ② 创建全零数组          ③ 上采样（插零）
    (n, 256)                (n, 1024)              (n, 1024)
                             全是 0
                                                   
[s₀, s₁, s₂, ...]  →  [0,0,0,...,0,0,0]  →  [s₀,0,0,0,s₁,0,0,0,s₂,0,0,0,...]
                                                        │
                                                        ▼ ④ RRC 卷积
                                              [~~连续平滑的波形~~~~~~~~~~]
                                                        │
                                                        ▼ 截取前1024点
                                              [~~最终基带信号~~]  (n, 1024)
```

---

## 数学公式

$$x_{\text{up}}[n] = \begin{cases} s_{n/\text{SPS}}, & n \mod \text{SPS} = 0 \\ 0, & \text{otherwise} \end{cases}$$

$$x_{\text{out}}[n] = (x_{\text{up}} * h_{\text{RRC}})[n] = \sum_{k} x_{\text{up}}[k] \cdot h_{\text{RRC}}[n-k]$$

---

## 二、关于默认参数 `NUM_SYM`

### 函数定义

```python
NUM_SYM = 256    # 模块级全局变量

def _rand_symbols(name, n_batch, n_sym=NUM_SYM):
    #                              ↑
    #                         默认值 = 256
```

### 调用时的三种情况

```python
# 情况 1: 不传第三个参数 → 自动用默认值 256
_rand_symbols('QPSK', 100)
# 等价于
_rand_symbols('QPSK', 100, 256)

# 情况 2: 显式传参 → 覆盖默认值
_rand_symbols('QPSK', 100, 128)
# n_sym = 128

# 情况 3: 关键字传参
_rand_symbols('QPSK', 100, n_sym=512)
# n_sym = 512
```

### ⚠️ 一个关键细节

默认值在**函数定义时**就确定了，不是调用时：

```python
NUM_SYM = 256

def _rand_symbols(name, n_batch, n_sym=NUM_SYM):   # 定义时 n_sym 绑定为 256
    ...

NUM_SYM = 512    # 之后修改 NUM_SYM

_rand_symbols('QPSK', 100)   # n_sym 仍然是 256！不是 512！
```

```
┌──────────────────────────────────────────────────┐
│  Python 默认参数的绑定时机：                       │
│                                                   │
│  def f(x=表达式):     ← 表达式在 def 执行时求值    │
│                         之后就固定了，不会再变      │
│                                                   │
│  所以 n_sym=NUM_SYM 在定义时就变成了 n_sym=256     │
│  后续修改 NUM_SYM 不会影响默认值                    │
└──────────────────────────────────────────────────┘
```

### 回答你的问题

> **是的，只要不需要修改 `NUM_SYM` 的值，调用时可以省略第三个参数：**

```python
# _gen_linear 中的调用：
syms = _rand_symbols(const_name, n)     # 只传 2 个参数
#                                        n_sym 自动 = 256 ✅
```

In [30]:
def gen_bpsk(n):    return _gen_linear('BPSK', n)
def gen_qpsk(n):    return _gen_linear('QPSK', n)
def gen_8psk(n):    return _gen_linear('8PSK', n)
def gen_16qam(n):   return _gen_linear('16QAM', n)
def gen_32qam(n):   return _gen_linear('32QAM', n)
def gen_64qam(n):   return _gen_linear('64QAM', n)
def gen_128qam(n):  return _gen_linear('128QAM', n)
def gen_256qam(n):  return _gen_linear('256QAM', n)

In [31]:
# B. OQPSK: Q 路半符号延迟
def gen_oqpsk(n):
    bi = 2.0 * np.random.randint(0, 2, (n, NUM_SYM)) - 1.0
    bq = 2.0 * np.random.randint(0, 2, (n, NUM_SYM)) - 1.0
    L = NUM_SYM * SPS

    up_i = np.zeros((n, L))
    up_i[:, ::SPS] = bi

    up_q = np.zeros((n, L))
    q_idx = np.arange(NUM_SYM) * SPS + SPS // 2
    q_idx = q_idx[q_idx < L]
    up_q[:, q_idx] = bq[:, :len(q_idx)]

    si = batch_fft_convolve(up_i, _RRC).real
    sq = batch_fft_convolve(up_q, _RRC).real
    return (si + 1j * sq)[:, :SIG_LEN]

# OQPSK 信号生成详解

---

## 一、OQPSK 与 QPSK 的区别

$$\boxed{\text{OQPSK} = \text{QPSK}，但 \text{Q 路延迟半个符号周期}}$$

```
QPSK:    I 路:  |──s₀──|──s₁──|──s₂──|──s₃──|
         Q 路:  |──s₀──|──s₁──|──s₂──|──s₃──|
                ↑      ↑      ↑      ↑
                I、Q 同时跳变

OQPSK:   I 路:  |──s₀──|──s₁──|──s₂──|──s₃──|
         Q 路:     |──s₀──|──s₁──|──s₂──|──s₃──|
                   ← T/2 →
                I、Q 交替跳变，不会同时跳
```

**好处**：避免 I、Q 同时跳变导致信号过零 → 包络波动更小 → 对 PA 更友好。

---

## 二、逐行代码解释

### ① 生成 I、Q 路独立的 BPSK 符号

```python
bi = 2.0 * np.random.randint(0, 2, (n, NUM_SYM)) - 1.0    # I 路
bq = 2.0 * np.random.randint(0, 2, (n, NUM_SYM)) - 1.0    # Q 路
```

| 步骤 | 值 |
|:-----|:---|
| `np.random.randint(0, 2, ...)` | 随机 `{0, 1}` |
| `2.0 * ... - 1.0` | 映射为 `{-1, +1}` |

$$b_I[k], \; b_Q[k] \in \{-1, +1\}, \quad k = 0,1,\ldots,255$$

> OQPSK 的 I 路和 Q 路各自独立做 BPSK（不像 QPSK 从 4 点星座统一取）。

---

### ② I 路上采样（正常位置）

```python
L = NUM_SYM * SPS     # 256 × 4 = 1024

up_i = np.zeros((n, L))
up_i[:, ::SPS] = bi
```

```
位置索引:  0  1  2  3  4  5  6  7  8  9  10 11 12 ...
up_i:    [b₀, 0, 0, 0, b₁, 0, 0, 0, b₂, 0, 0, 0, b₃, ...]
          ↑           ↑           ↑           ↑
         [0]         [4]         [8]        [12]
```

符号放在 $n \times \text{SPS}$ 的整数倍位置，**与普通 QPSK 完全相同**。

---

### ③ Q 路上采样（延迟半符号）⭐ 关键差异

```python
q_idx = np.arange(NUM_SYM) * SPS + SPS // 2
#       [0, 1, 2, ...255] × 4   +    2
#     = [0, 4, 8, ...1020]      +    2
#     = [2, 6, 10, ...1022]
q_idx = q_idx[q_idx < L]    # 确保不越界

up_q = np.zeros((n, L))
up_q[:, q_idx] = bq[:, :len(q_idx)]
```

| 代码 | 计算 | 结果 |
|:-----|:-----|:-----|
| `np.arange(NUM_SYM) * SPS` | $k \times 4$ | `[0, 4, 8, 12, ...]` |
| `+ SPS // 2` | $+ 4//2 = +2$ | `[2, 6, 10, 14, ...]` |

```
位置索引:  0  1  2  3  4  5  6  7  8  9  10 11 12 ...
up_i:    [b₀, 0, 0, 0, b₁, 0, 0, 0, b₂, 0, 0, 0, b₃, ...]
up_q:    [ 0, 0, b₀,0,  0, 0, b₁,0,  0, 0, b₂,0,  0, ...]
                 ↑           ↑           ↑
                [2]         [6]        [10]
```

**Q 路每个符号比 I 路延迟了 2 个采样点 = 半个符号周期：**

$$\text{延迟} = \frac{\text{SPS}}{2} = \frac{4}{2} = 2 \text{ 个采样点} = \frac{T}{2}$$

对比 I、Q 的符号位置：

```
I 路:  位置 [0,  4,  8,  12, 16, ...]   ← 整符号时刻
Q 路:  位置 [2,  6,  10, 14, 18, ...]   ← 延迟 T/2
              ↑
           差 2 个采样点 = T/2
```

---

### ④ 分别 RRC 脉冲成形

```python
si = batch_fft_convolve(up_i, _RRC).real    # I 路成形，取实部
sq = batch_fft_convolve(up_q, _RRC).real    # Q 路成形，取实部
```

> 因为 I、Q 各自是实数序列（±1），RRC 也是实系数滤波器，卷积结果本应是实数。取 `.real` 是为了消除浮点运算中微小的虚部残余。

---

### ⑤ 合成复信号

```python
return (si + 1j * sq)[:, :SIG_LEN]
```

$$\boxed{x[n] = \underbrace{s_I[n]}_{\text{I 路}} + j \cdot \underbrace{s_Q[n]}_{\text{Q 路（延迟 T/2）}}}$$

截取前 `SIG_LEN=1024` 个点。

---

## 三、完整流程图

```
I 路:  随机{±1}  →  [b₀, 0,0,0, b₁, 0,0,0, ...]  →  RRC滤波  →  sI[n]
       (256个)      ↑ 放在 0,4,8,12...                          ↘
                                                                  → sI + j·sQ → 输出
Q 路:  随机{±1}  →  [0,0, b₀, 0, 0,0, b₁, 0, ...]  →  RRC滤波  →  sQ[n]
       (256个)        ↑ 放在 2,6,10,14...                       ↗
                      ← 延迟 T/2 →
```

---

## 四、数学公式

### QPSK 信号

$$x_{\text{QPSK}}(t) = \sum_k b_I[k] \cdot p(t - kT) + j \sum_k b_Q[k] \cdot p(t - kT)$$

### OQPSK 信号

$$\boxed{x_{\text{OQPSK}}(t) = \sum_k b_I[k] \cdot p(t - kT) + j \sum_k b_Q[k] \cdot p\!\left(t - kT - \frac{T}{2}\right)}$$

唯一区别：Q 路的脉冲延迟了 $T/2$。

---

## 五、为什么要这样做？

### QPSK 的问题

I、Q 同时跳变时，信号可能**穿过原点**：

```
Q ↑
  │  ·       ·
  │     ╲ ╱        ← 信号轨迹穿过原点
  │      ╳         ← 包络瞬间降为 0
  │     ╱ ╲
  │  ·       ·
  └──────────→ I
```

包络突降 → PA 工作在非线性区 → 信号失真。

### OQPSK 的解决

I、Q **不同时跳变** → 信号**不穿过原点**：

```
Q ↑
  │  ·       ·
  │  │       │     ← 信号轨迹只沿 I 或 Q 轴移动
  │  │       │     ← 不穿过原点
  │  ·───────·     ← 包络波动更小
  └──────────→ I
```

| 对比 | QPSK | OQPSK |
|:-----|:-----|:------|
| 最大相位跳变 | ±180° | ±90° |
| 是否过零 | ✅ 可能 | ❌ 不会 |
| 包络波动 | 大 | 小 |
| PA 友好度 | 一般 | 更好 |
| 频谱效率 | 相同 | 相同 |

# `up_q[:, q_idx] = bq[:, :len(q_idx)]` 详解

---

## 先回顾上下文

```python
# 假设 n=3, NUM_SYM=256, SPS=4, L=1024

up_q = np.zeros((3, 1024))              # 全零数组

q_idx = np.arange(256) * 4 + 2          # [2, 6, 10, 14, ..., 1022]
q_idx = q_idx[q_idx < 1024]             # 过滤越界（本例全部 < 1024，共 256 个）

bq    # shape = (3, 256)，每个值 ∈ {-1, +1}
```

---

## 拆解为两部分

### 左边：`up_q[:, q_idx]` — 在哪里写？

```python
up_q[:, q_idx]
#  ↑      ↑
# 所有行  指定列
```

`q_idx = [2, 6, 10, 14, ..., 1022]`

意思是：选出 `up_q` 的**所有行**的**第 2、6、10、14、...、1022 列**

```
up_q 原始（全零）:

列索引:   0  1  [2]  3  4  5  [6]  7  8  9 [10] 11  12 13 [14] ...
行0:      0  0   0   0  0  0   0   0  0  0   0   0   0  0   0  ...
行1:      0  0   0   0  0  0   0   0  0  0   0   0   0  0   0  ...
行2:      0  0   0   0  0  0   0   0  0  0   0   0   0  0   0  ...
               ↑          ↑           ↑            ↑
           这些位置被选中，准备写入
```

`up_q[:, q_idx]` 的 shape = `(3, 256)` — 选出了 3 行 × 256 个列位置

---

### 右边：`bq[:, :len(q_idx)]` — 写入什么？

```python
bq[:, :len(q_idx)]
#  ↑      ↑
# 所有行  前 len(q_idx)=256 个符号
```

`bq` 的 shape = `(3, 256)`，取前 256 个 → 实际就是**整个 bq**

> 为什么要写 `:len(q_idx)` 而不是直接用 `bq`？因为 `q_idx` 经过了越界过滤，**可能少于 256 个**。要保证左右 shape 匹配。

---

## 赋值过程（用小例子）

假设 `n=1, NUM_SYM=5, SPS=4, L=20`：

```python
bq = [[-1, +1, -1, +1, -1]]          # shape = (1, 5)

q_idx = [2, 6, 10, 14, 18]           # 5 个位置

up_q = [[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]]   # shape = (1, 20)
```

执行 `up_q[:, q_idx] = bq[:, :5]`：

```
                    逐个对应填入
                    
q_idx:     [2,     6,    10,    14,    18]
bq:        [-1,   +1,    -1,   +1,    -1]
             ↓      ↓      ↓      ↓      ↓

列索引:  0  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19
up_q:   [0, 0, -1, 0, 0, 0,+1, 0, 0, 0, -1, 0, 0, 0,+1, 0, 0, 0, -1, 0]
              ↑           ↑            ↑            ↑            ↑
             [2]         [6]         [10]         [14]         [18]
```

---

## 对比 I 路

```
I 路:  up_i[:, ::4]   = bi     ← 放在 [0, 4, 8, 12, 16]
Q 路:  up_q[:, q_idx] = bq     ← 放在 [2, 6, 10, 14, 18]

列索引:  0  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19
up_i:  [I₀, 0, 0, 0, I₁, 0, 0, 0, I₂, 0, 0, 0, I₃, 0, 0, 0, I₄, 0, 0, 0]
up_q:  [ 0, 0,Q₀, 0,  0, 0,Q₁, 0,  0, 0,Q₂, 0,  0, 0,Q₃, 0,  0, 0,Q₄, 0]
            ← 2 →
        I在0位置   Q在2位置：延迟了 T/2
```

---

## 为什么需要 `:len(q_idx)`？

### 边界情况

如果参数不凑巧，最后一个 `q_idx` 可能越界：

```python
# 假设 NUM_SYM=5, SPS=4, L=18（不是整倍数）
q_idx = [2, 6, 10, 14, 18]
q_idx = q_idx[q_idx < 18]       # 18 被过滤！
# q_idx = [2, 6, 10, 14]        # 只剩 4 个

# bq shape = (n, 5)，但 q_idx 只有 4 个位置
# 必须截取 bq 的前 4 个才能匹配：
up_q[:, q_idx] = bq[:, :4]      # 左(n,4) = 右(n,4) ✅
# 如果写 bq 全部：
up_q[:, q_idx] = bq             # 左(n,4) = 右(n,5) ❌ 报错！
```

```
┌──────────────────────────────────────────┐
│  :len(q_idx) 是一个安全措施               │
│                                          │
│  正常情况: len(q_idx) = NUM_SYM = 256    │
│           → bq[:, :256] = 整个 bq        │
│                                          │
│  边界情况: len(q_idx) < NUM_SYM          │
│           → 截取 bq 前几个，匹配长度      │
└──────────────────────────────────────────┘
```

---

## 一句话总结

> **`up_q[:, q_idx] = bq[:, :len(q_idx)]`：把 Q 路符号放到延迟 T/2 的位置上（第 2、6、10、14... 列），同时用 `:len(q_idx)` 确保左右数量匹配不报错。**

In [32]:
# C. π/4-DQPSK: 差分编码 + π/4 旋转
def gen_pi4dqpsk(n):
    dphi = np.array([np.pi / 4, 3 * np.pi / 4, -3 * np.pi / 4, -np.pi / 4])
    idx = np.random.randint(0, 4, (n, NUM_SYM))
    phase = np.cumsum(dphi[idx], axis=1)  # 矢量化差分累积
    syms = np.exp(1j * phase)

    up = np.zeros((n, NUM_SYM * SPS), dtype=np.complex128)
    up[:, ::SPS] = syms
    return batch_fft_convolve(up, _RRC)[:, :SIG_LEN]

# π/4-DQPSK 详解

---

## 一、什么是 π/4-DQPSK？

### 名字拆解

$$\underbrace{\pi/4}_{\text{每次旋转}45°}\text{-}\underbrace{D}_{\text{差分(Differential)}}\underbrace{QPSK}_{\text{4相移键控}}$$

### 核心思想

> **信息不编码在"绝对相位"中，而是编码在"相邻符号的相位差"中。**

```
普通 QPSK:   信息 → 绝对相位 θₖ
π/4-DQPSK:   信息 → 相位变化量 Δθₖ = θₖ - θₖ₋₁
```

---

## 二、数学公式

### 1. 差分相位映射

每个符号携带 2 bit，映射为 4 种**相位增量**：

$$\Delta\phi \in \left\{\frac{\pi}{4},\; \frac{3\pi}{4},\; -\frac{3\pi}{4},\; -\frac{\pi}{4}\right\}$$

| 2 bit 输入 | $\Delta\phi$ | 角度 |
|:----------:|:------------:|:----:|
| 00 | $+\pi/4$ | $+45°$ |
| 01 | $+3\pi/4$ | $+135°$ |
| 11 | $-3\pi/4$ | $-135°$ |
| 10 | $-\pi/4$ | $-45°$ |

### 2. 累积相位

$$\boxed{\theta[k] = \theta[k-1] + \Delta\phi[k] = \sum_{i=1}^{k} \Delta\phi[i]}$$

### 3. 输出符号

$$\boxed{s[k] = e^{j\,\theta[k]}}$$

---

## 三、代码逐行讲解

### ① 定义 4 种相位增量

```python
dphi = np.array([np.pi/4, 3*np.pi/4, -3*np.pi/4, -np.pi/4])
#                  45°      135°       -135°       -45°
```

$$\Delta\phi \in \{+45°,\; +135°,\; -135°,\; -45°\}$$

### ② 随机选择相位增量（模拟随机 2bit 数据）

```python
idx = np.random.randint(0, 4, (n, NUM_SYM))
# shape = (n, 256)
# 每个值 ∈ {0, 1, 2, 3}
```

```python
dphi[idx]    # 花式索引，将 {0,1,2,3} 映射为对应的 Δφ 值
# shape = (n, 256)
```

例如一条信号：

```
idx:       [0,    2,    1,    3,    0,   ...]
dphi[idx]: [π/4, -3π/4, 3π/4, -π/4, π/4, ...]
            45°  -135°  135°  -45°  45°
```

### ③ 累积求和 → 绝对相位 ⭐ 核心步骤

```python
phase = np.cumsum(dphi[idx], axis=1)
```

`np.cumsum` 沿 `axis=1`（列方向）做**前缀和**：

$$\theta[k] = \sum_{i=0}^{k} \Delta\phi[i]$$

具体例子：

```
Δφ:     [  π/4,   -3π/4,    3π/4,   -π/4,    π/4  ]

cumsum: [  π/4,   π/4-3π/4, ...,    ...,     ...   ]
      = [  π/4,   -π/2,     π/4,    0,       π/4   ]
         ↑        ↑         ↑       ↑        ↑
        θ[0]     θ[1]      θ[2]   θ[3]     θ[4]

验证:
θ[0] = π/4                           = 45°
θ[1] = π/4 + (-3π/4)     = -π/2     = -90°
θ[2] = -π/2 + 3π/4       = π/4      = 45°
θ[3] = π/4 + (-π/4)      = 0        = 0°
θ[4] = 0 + π/4           = π/4      = 45°
```

### ④ 生成复数符号

```python
syms = np.exp(1j * phase)    # shape = (n, 256)
```

$$s[k] = e^{j\,\theta[k]}$$

```
θ:     [  π/4,   -π/2,   π/4,    0,     π/4  ]
         ↓        ↓       ↓      ↓       ↓
s:     [e^jπ/4, e^-jπ/2, e^jπ/4, 1,  e^jπ/4]
     = [(1+j)/√2,  -j,  (1+j)/√2, 1, (1+j)/√2]
```

### ⑤ 上采样 + RRC 成形（与普通线性调制相同）

```python
up = np.zeros((n, NUM_SYM * SPS), dtype=np.complex128)
up[:, ::SPS] = syms
return batch_fft_convolve(up, _RRC)[:, :SIG_LEN]
```

---

## 四、星座图的变化过程

### 关键特性：星座在两组之间交替

```
奇数时刻的星座（旋转 π/4）          偶数时刻的星座（标准 QPSK）

    Q ↑                                Q ↑
      |    ·                             ·       ·
      |  ╱                               |
·─────┼─────·  → I                  ─────┼─────  → I
      |  ╲                               |
      |    ·                             ·       ·

  {π/4, 3π/4, -3π/4, -π/4}          {0, π/2, π, -π/2}
```

合并后，所有可能的星座点：

```
         Q ↑
     ·   ·   ·
      ╲  |  ╱
       ╲ | ╱
  · ────╋──── · → I
       ╱ | ╲
      ╱  |  ╲
     ·   ·   ·

    共 8 个可能位置
```

> 虽然有 8 个可能位置，但每个时刻**只用其中 4 个**，并且相邻时刻使用的 4 个点互相旋转 45°。

---

## 五、为什么每次旋转 ±45° 或 ±135°？

### 相位跳变分析

| | QPSK | π/4-DQPSK |
|:--|:-----|:----------|
| 可能的相位跳变 | $0°, ±90°, 180°$ | $±45°, ±135°$ |
| 最大跳变 | **180°**（过零） | **135°**（不过零） |
| 信号过零？ | ✅ 可能 | ❌ **永远不会** |

```
QPSK 180° 跳变:              π/4-DQPSK 最大 135° 跳变:

Q ↑                           Q ↑
  |  ·                          |     ·
  |    ╲                        |    ╱
  |      ╲ 穿过原点              |   ╱  不穿过原点
  |        ╲                    |  ╱
  ·──────────→ I               · ╱────────→ I
```

**不过零 → 包络波动小 → PA 更高效。**

---

## 六、"差分"的意义

### 接收端解调

普通 QPSK 需要**载波相位同步**（知道绝对相位）才能解调。

π/4-DQPSK 只需要比较**相邻两个符号的相位差**：

$$\hat{\Delta\phi}[k] = \angle\left(\frac{r[k]}{r[k-1]}\right) = \angle\big(r[k] \cdot r^*[k-1]\big)$$

```
发射端:  数据 → Δφ → 累加 → 绝对相位 → 发射
接收端:  接收 → 相邻相位差 → 查表 → 数据

         不需要知道绝对相位！
         即使载波有固定频偏，差分也能消除！
```

| 对比 | QPSK | π/4-DQPSK |
|:-----|:-----|:----------|
| 编码 | 绝对相位 | 相位差 |
| 需要相位同步？ | ✅ 必须 | ❌ 不需要 |
| 抗频偏能力 | 弱 | **强** |
| 代价 | — | 约 3dB SNR 损失 |

---

## 七、完整流程图

```
随机2bit数据                  相位增量                 累积相位
  [0,1,2,3,...]     →     [π/4, 3π/4, -3π/4,...]  →  cumsum
  (n, 256)                  (n, 256)                  (n, 256)
                                                         │
                                                         ▼
复数符号                    上采样                     RRC滤波
  e^(jθ[k])        →    [s₀,0,0,0,s₁,0,0,0,...]  →  平滑波形
  (n, 256)               (n, 1024)                   (n, 1024)
```

---

## 八、总结公式

$$\boxed{
\begin{aligned}
&\textbf{相位增量映射:} \quad \Delta\phi[k] \in \left\{\pm\frac{\pi}{4},\; \pm\frac{3\pi}{4}\right\} \\[6pt]
&\textbf{差分累积:} \quad \theta[k] = \sum_{i=0}^{k} \Delta\phi[i] \\[6pt]
&\textbf{输出符号:} \quad s[k] = e^{\,j\,\theta[k]} \\[6pt]
&\textbf{解调:} \quad \hat{\Delta\phi}[k] = \angle\!\big(s[k] \cdot s^*[k\!-\!1]\big)
\end{aligned}
}$$

In [33]:
# D. 2ASK (OOK) 
# 值为[0, 1]区别于[-1,1]BPSK
def gen_2ask(n):
    bits = np.random.randint(0, 2, (n, NUM_SYM)).astype(np.float64)
    up = up.zeros((n, NUM_SYM * SPS), dtype=np.complex128)
    up[:, ::SPS] = bits
    return batch_fft_convolve(up, _RRC)[:, :SIG_LEN]

In [36]:
# E. CPFSK 族 (矢量化，无逐符号循环)
def _gen_cpfsk(n, M, h_mod):
    """
    通用连续相位频移键控 (矢量化)

    M:     调制阶数 (2, 4)
    h_mod: 调制指数
    """
    syms = np.random.randint(0, M, (n, NUM_SYM))
    levels = 2.0 * np.arange(M) - (M - 1)  # [-1,1] / [-3,-1,1,3]
    freq_sym = levels[syms]  # (N, NUM_SYM)

    # 归一化频偏: phase_per_symbol = π · h · a_k
    # phase_per_sample = π · h · a_k / SPS
    freq_per_sample = h_mod * freq_sym / (2.0 * SPS)  # 归一化到 cycles/sample

    # 上采样: repeat
    # 此处是矩形脉冲
    freq_up = np.repeat(freq_per_sample, SPS, axis = 1)[:, :SIG_LEN]

    # 累积相位
    phase = 2 * np.pi * np.cumsum(freq_up, axis = 1)
    return np.exp(1j * phase)

In [37]:
def gen_4fsk(n):
    """4FSK: 使用调制指数 h=1/3"""
    M = 4
    h_mod = 1.0 / 3.0

    syms = np.random.randint(0, M, (n, NUM_SYM))
    levels = 2.0 * np.arange(M) - (M - 1) # [-3, -1, 1, 3]
    freq_sym = levels[syms]

    freq_per_sample = h_mod * freq_sym / (2.0 * SPS)
    freq_up = np.repeat(freq_per_sample, SPS, axis=1)[:, ::SIG_LEN]
    phase = 2 * np.pi *  np.cumsum(freq_up, axis=1)
    return np.exp(1j * phase)

# CPFSK 信号生成详解

---

## 一、什么是 CPFSK？

$$\text{CPFSK} = \text{Continuous Phase} + \text{Frequency Shift Keying}$$

- **FSK**：用不同频率表示不同符号
- **Continuous Phase**：频率切换时**相位不跳变**，保持连续

```
普通 FSK:                    CPFSK:

相位                         相位
  ↑    ╱╲                     ↑    ╱╲
  |   ╱  ╲   ╱╲               |   ╱  ╲   ╱╲
  |  ╱  ↑ ╲ ╱  ╲              |  ╱    ╲╱╱  ╲
  | ╱ 跳变  ╲    ╲             | ╱   ↑       ╲
  +──────────────→ t          +──连续──────────→ t
     相位不连续                    相位连续
```

---

## 二、数学公式

### CPFSK 信号

$$\boxed{x(t) = e^{\,j\,\phi(t)}}$$

### 瞬时相位

$$\boxed{\phi(t) = 2\pi \int_0^{t} f(\tau)\,d\tau}$$

### 瞬时频率（第 k 个符号周期内）

$$f(t) = \frac{h \cdot a_k}{2T}, \quad kT \le t < (k+1)T$$

其中：
- $h$ = 调制指数（`h_mod`）
- $a_k$ = 第 k 个符号的频率电平
- $T$ = 符号周期

### 每个符号的相位增量

$$\Delta\phi_k = \pi \cdot h \cdot a_k$$

---

## 三、代码逐行讲解

### ① 随机选择符号

```python
syms = np.random.randint(0, M, (n, NUM_SYM))
# M=2: 值 ∈ {0, 1}
# M=4: 值 ∈ {0, 1, 2, 3}
```

### ② 映射为频率电平

```python
levels = 2.0 * np.arange(M) - (M - 1)
freq_sym = levels[syms]
```

| M | `np.arange(M)` | `levels` | 含义 |
|:-:|:-:|:-:|:-:|
| 2 | `[0, 1]` | `[-1, +1]` | 两个频率：低、高 |
| 4 | `[0, 1, 2, 3]` | `[-3, -1, +1, +3]` | 四个频率 |

```python
freq_sym = levels[syms]    # 花式索引，shape = (n, NUM_SYM)
```

例如 M=2：

```
syms:     [0,    1,    1,    0,    1,   ...]
levels:   [-1,  +1]
freq_sym: [-1,  +1,   +1,   -1,   +1,  ...]
           低频  高频  高频  低频  高频
```

### ③ 计算每采样点的频率（归一化）⭐

```python
freq_per_sample = h_mod * freq_sym / (2.0 * SPS)
```

推导过程：

$$\underbrace{\Delta\phi_k}_{\text{每符号相位增量}} = \pi \cdot h \cdot a_k$$

每个符号有 SPS 个采样点，所以**每采样点的相位增量**：

$$\Delta\phi_{\text{sample}} = \frac{\pi \cdot h \cdot a_k}{\text{SPS}}$$

转化为**每采样点的频率**（cycles/sample）：

$$f_{\text{sample}} = \frac{\Delta\phi_{\text{sample}}}{2\pi} = \frac{h \cdot a_k}{2 \cdot \text{SPS}}$$

对应代码：

```python
freq_per_sample = h_mod * freq_sym / (2.0 * SPS)
#                    h   ×   aₖ    / (2 × SPS)
```

| 变量 | 数学 | 单位 |
|:-----|:-----|:-----|
| `freq_sym` | $a_k$ | 无量纲电平 |
| `freq_per_sample` | $\frac{h \cdot a_k}{2 \cdot \text{SPS}}$ | cycles/sample |

### ④ 上采样：每符号重复 SPS 次

```python
freq_up = np.repeat(freq_per_sample, SPS, axis=1)[:, :SIG_LEN]
```

`np.repeat` 把每个值重复 SPS=4 次：

```
freq_per_sample: [f₀,         f₁,         f₂,        ...]
                  ↓            ↓            ↓
freq_up:         [f₀,f₀,f₀,f₀, f₁,f₁,f₁,f₁, f₂,f₂,f₂,f₂, ...]
                  ← SPS=4 →    ← SPS=4 →

shape: (n, 256) → (n, 1024)
```

**注意：这里用 `repeat`（重复）而不是插零！**

```
对比:
线性调制（QAM/PSK）:  [s₀, 0, 0, 0, s₁, 0, 0, 0, ...]   ← 插零 + RRC 滤波
CPFSK:                [f₀,f₀,f₀,f₀, f₁,f₁,f₁,f₁, ...]   ← 重复（矩形频率脉冲）
```

> CPFSK 不需要 RRC 脉冲成形，因为它的频率脉冲本身就是**矩形**。

### ⑤ 累积相位

```python
phase = 2 * np.pi * np.cumsum(freq_up, axis=1)
```

$$\phi[n] = 2\pi \sum_{i=0}^{n} f_{\text{sample}}[i]$$

这就是对频率做**离散积分**得到相位：

```
freq_up: [f₀, f₀, f₀, f₀, f₁, f₁, f₁, f₁, ...]

cumsum:  [f₀, 2f₀, 3f₀, 4f₀, 4f₀+f₁, 4f₀+2f₁, ...]
          × 2π
phase:   [2πf₀, 4πf₀, 6πf₀, 8πf₀, ...]
              ↑
          相位线性增长 = 恒定频率
          每个符号内频率恒定 → 相位是斜线
          符号边界处频率改变 → 斜率改变，但相位连续！
```

### ⑥ 生成复信号

```python
return np.exp(1j * phase)
```

$$x[n] = e^{j\,\phi[n]}$$

恒包络：$|x[n]| = 1$

---

## 四、相位轨迹示意图

以 2FSK, h=1.0 为例，$a_k \in \{-1, +1\}$：

```
每符号相位增量 = π × h × aₖ = π × 1 × (±1) = ±π

phase
  ↑
  │        ╱╲
  │       ╱  ╲        ╱
  │      ╱    ╲      ╱
  │     ╱      ╲    ╱
  │    ╱        ╲  ╱
  │   ╱          ╲╱
  │  ╱
  +─────────────────────→ 采样点
     aₖ: +1  +1  -1  -1  +1  +1
     
  斜率 ∝ 频率
  正斜率 = 高频（aₖ=+1）
  负斜率 = 低频（aₖ=-1）
  转折处相位连续！← CPFSK 的关键
```

---

## 五、MSK 作为 CPFSK 的特例

代码中调用方式：

```python
def gen_2fsk(n):    return _gen_cpfsk(n, M=2, h_mod=1.0)
def gen_4fsk(n):    return _gen_cpfsk(n, M=4, h_mod=1.0)
def gen_msk(n):     return _gen_cpfsk(n, M=2, h_mod=0.5)    # ← MSK！
```

| 调制 | M | h | 每符号相位增量 |
|:-----|:-:|:-:|:-------------|
| 2FSK | 2 | 1.0 | $\pm\pi$ |
| 4FSK | 4 | 1.0 | $\{-3\pi, -\pi, +\pi, +3\pi\}$ |
| **MSK** | **2** | **0.5** | $\pm\pi/2$（**最小频移**） |

> MSK = 调制指数 h=0.5 的 2-CPFSK，这是能保证正交性的**最小**频率间隔，因此叫 Minimum Shift Keying。

---

## 六、为什么 CPFSK 不用 RRC？

| | 线性调制（PSK/QAM） | CPFSK |
|:--|:-------------------|:------|
| 脉冲成形 | 需要 RRC 滤波 | **不需要** |
| 原因 | 符号是复数脉冲，需要平滑 | 频率脉冲本身是矩形，相位自然连续 |
| 恒包络？ | ❌ 不是 | ✅ 恒包络 |
| 上采样方式 | 插零 + 滤波 | **重复**（repeat） |

```
线性调制:  插零 [s₀,0,0,0,s₁,0,0,0] → RRC滤波 → 平滑波形
CPFSK:     重复 [f₀,f₀,f₀,f₀,f₁,f₁,f₁,f₁] → 累积相位 → exp(jφ) → 天然平滑
```

---

## 七、完整流程总结

```
随机符号            频率电平            每采样频率             上采样
{0,..,M-1}    →    {±1} or {±3,±1}  →  h·aₖ/(2·SPS)    →   repeat×SPS
(n, 256)           (n, 256)            (n, 256)             (n, 1024)
                                                               │
                                                               ▼
输出信号         ←     e^(jφ)      ←     2π·cumsum        ← 频率序列
(n, 1024)            恒包络              离散积分          (n, 1024)
```

$$\boxed{
\begin{aligned}
&a_k \in \{-(M\!-\!1),\ldots,-1,+1,\ldots,+(M\!-\!1)\} \\[4pt]
&f[n] = \frac{h \cdot a_k}{2 \cdot \text{SPS}}, \quad kT \le n < (k\!+\!1)T \\[4pt]
&\phi[n] = 2\pi \sum_{i=0}^{n} f[i] \\[4pt]
&x[n] = e^{\,j\,\phi[n]}
\end{aligned}
}$$

In [38]:
def gen_2fsk(n):  return _gen_cpfsk(n, M=2, h_mod=1.0)
def gen_msk(n):   return _gen_cpfsk(n, M=2, h_mod=0.5)

# 核心就是把 `np.repeat`（矩形脉冲）替换为高斯滤波脉冲

---

## 一、`np.repeat` 的本质 = 矩形频率脉冲

```python
# MSK / CPFSK 当前代码
freq_up = np.repeat(freq_per_sample, SPS, axis=1)
```

```
freq_per_sample: [ f₀,          f₁,          f₂,       ...]

np.repeat:       [ f₀, f₀, f₀, f₀, f₁, f₁, f₁, f₁, f₂, f₂, f₂, f₂, ...]
                  ├── 矩形脉冲 ──┤├── 矩形脉冲 ──┤
                  
                  频率值在符号内恒定，边界处突变
```

等效于与**矩形脉冲卷积**：

$$f_{\text{up}}(t) = \sum_k f_k \cdot \underbrace{\text{rect}\!\left(\frac{t - kT}{T}\right)}_{\text{np.repeat 做的事}}$$

---

## 二、GMSK 的修改方式

### 方法：`np.repeat` → 高斯卷积

```python
def _gen_gmsk(n, BT=0.3)

```

---

## 三、修改前后对比

### 频率波形对比

```
MSK（np.repeat = 矩形）:

freq
  ↑
  │  ┌──────┐          ┌──────┐
  │  │      │          │      │
──┼──┘      └──────────┘      └──── → 采样点
  │         ┌──────────┐
  │         │          │
  ↓         └──────────┘

  频率在符号边界处「突变」


GMSK（高斯卷积后）:

freq
  ↑
  │    ╭────╮            ╭────╮
  │   ╱      ╲          ╱      ╲
──┼──╯        ╲────────╯        ╲──── → 采样点
  │            ╲────────╮
  │             ╲      ╱
  ↓              ╰────╯

  频率在符号边界处「平滑过渡」
```

### 相位轨迹对比

```
MSK:                           GMSK:

phase                          phase
  ↑      ╱╲                      ↑      ╱╲
  │     ╱  ╲    ╱                │    ╭╯  ╰╮  ╭╯
  │    ╱    ╲  ╱                 │   ╱      ╲╱
  │   ╱      ╲╱                  │  ╱
  │  ╱                           │ ╱
  +──────────→ t                 +──────────→ t
  
  折线（斜率突变）                 平滑曲线（斜率渐变）
```

---

## 四、核心差异只有一步

```python
# MSK
freq_up = np.repeat(...)                              # 矩形

# GMSK（修改这一步）
freq_up = batch_fft_convolve(np.repeat(...), h_gauss) # 矩形 * 高斯
#                                            ↑
#                                       高斯滤波器
```

其余代码**完全不变**：

```python
phase = 2 * np.pi * np.cumsum(freq_up, axis=1)    # 相同
return np.exp(1j * phase)                           # 相同
```

---

## 五、完整对比表

| | MSK | GMSK |
|:--|:----|:-----|
| 调制阶数 M | 2 | 2 |
| 调制指数 h | 0.5 | 0.5 |
| 频率脉冲 | $\text{rect}(t/T)$ | $\text{rect}(t/T) * g(t)$ |
| 代码差异 | `np.repeat` | `np.repeat` + **高斯卷积** |
| 频率跳变 | 突变 | 平滑 |
| 频谱 | 较宽（旁瓣高） | 窄（旁瓣低） |
| ISI | 无 | 有（脉冲展宽） |
| 恒包络 | ✅ | ✅ |
| 参数 | 无额外参数 | **BT**（带宽时间积） |

---

## 六、BT 参数的效果

```python
BT = ∞     →  高斯脉冲 = 冲激 → 不滤波 → GMSK 退化为 MSK
BT = 0.5   →  轻度平滑
BT = 0.3   →  GSM 标准 ★
BT = 0.2   →  强平滑，ISI 严重
```

---

## 七、一句话总结

$$\boxed{\text{GMSK} = \text{MSK 代码中，将} \;\texttt{np.repeat}\; \text{的输出再通过一个高斯低通滤波器}}$$

```
MSK:   freq_sym → np.repeat（矩形）→ cumsum → exp(jφ)
                       ↓ 加一步
GMSK:  freq_sym → np.repeat（矩形）→ 高斯卷积 → cumsum → exp(jφ)
                                      ↑
                                   唯一修改
```

In [40]:
def _gen_gmsk(n, BT=0.3):
    """
    GMSK = MSK (h=0.5, M=2) + 高斯频率脉冲
    """
    M, h_mod = 2, 0.5
    syms = np.random.randint(0, M, (n, NUM_SYM))
    levels = 2.0 * np.arange(M) - (M - 1)           # [-1, +1]
    freq_sym = levels[syms]

    freq_per_sample = h_mod * freq_sym / (2.0 * SPS)

    # GMSK：矩形 → 高斯滤波
    # Step 1: 先 repeat（矩形脉冲）
    freq_rect = np.repeat(freq_per_sample, SPS, axis=1)

    # Step 2: 构造高斯滤波器
    T = SPS  # 符号周期（采样点数）
    B = BT / T * SPS   # 带宽（采样率归一化）
    t_gauss = np.arange(-2*SPS, 2*SPS + 1)   # 截断到 ±2 符号
    h_guass = (2 * np.pi * B / np.sqrt(2 * np.pi * np.log(2))) * \
                np.exp(-2 * (np.pi * B * t_gauss / SPS)**2 / np.log(2)) # 归一化
    h_guass /= h_guass.sum()

    # Step 3: 卷积（矩形 * 高斯 = 高斯平滑后的脉冲）
    freq_up = batch_fft_convolve(freq_rect, h_guass).real[:, :SIG_LEN]

    phase = 2 * np.pi * np.cumsum(freq_up, axis=1)
    return np.exp(1j * phase)

In [39]:
# 8.生成器注册表
GENERATORS = {
    'BPSK': gen_bpsk,    'QPSK': gen_qpsk,      '8PSK': gen_8psk,
    'OQPSK': gen_oqpsk,  '2FSK': gen_2fsk,      '4FSK': gen_4fsk,
    '16QAM': gen_16qam,  '2ASK': gen_2ask,      'Pi4DQPSK': gen_pi4dqpsk,
    '64QAM': gen_64qam,  '32QAM': gen_32qam,    '128QAM': gen_128qam,
    '256QAM': gen_256qam, 'MSK': gen_msk,
}

In [41]:
# 9.主生成
def generate_all(cfg=CONFIG):
    np.random.seed(cfg['seed'])

    channel = ChannelModel(cfg)
    impairments = RFImpairments(cfg)

    classes = cfg['classes']
    snr_list = cfg['snr_list_db']
    n_per = cfg['n_per_class_per_snr']

    all_iq, all_y, all_snr = [], [], []
    total = len(classes) * len(snr_list)
    pbar = tqdm(total=total, desc='Generating', ncols=90)

    for cls_idx, cls_name in enumerate(classes):
        gen_fn = GENERATORS[cls_name]
        
        for snr_db in snr_list:
            # ① 生成干净基带信号
            clean = gen_fn(n_per)   # (N, SIG_LEN) complex

            # ② TX 损伤 (PA)
            clean = impairments.apply_tx(clean)

            # ③ 信道 (衰落 + AWGN)
            noisy = channel.apply(clean, snr_db)

            # ④ RX 损伤 (CFO, 相噪, IQ不平衡, DC, 时偏)
            noisy = impairments.apply_rx(noisy)

            # ⑤ 转 IQ: (N, 2, L)
            iq = np.stack([noisy.real, noisy.imag], axis=1).astype(np.float32)

            all_iq.append(iq)
            all_y.extend([cls_idx] * n_per)
            all_snr.extend([float(snr_db)] * n_per)
            
            pbar.set_postfix(cls=cls_name, snr=f'{snr_db:+d}dB')
            pbar.update(1)
    
    pbar.close()

    X = np.concatenate(all_iq, axis = 0)
    Y = np.array(all_y, dtype=np.int64)
    SNR = np.array(all_snr, dtype=np.float32)

    print(f'\n 总样本：{len(Y):,}  X: {X.shape}  '
          f'SNR: {SNR.min():.0f}~{SNR.max():.0f} dB')
    return X, Y, SNR

In [42]:
# 10. 保存  
def split_and_save(X, Y, SNR, cfg=CONFIG):
    os.makedirs(cfg['save_dir'], exist_ok=True)
    idx = np.arange(len(Y))
    i_tr, i_te = train_test_split(idx, test_size=1 - cfg['train_ratio'], random_state=cfg['seed'], stratify=Y)

    path = os.path.join(cfg['save_dir'], 'SionnaX.npz')
    np.savez_compressed(path,
                        X_train=X[i_tr], Y_train=Y[i_tr],
                        X_test=X[i_te], Y_test=Y[i_te],
                        snr_test=SNR[i_te],
                        classes=np.array(cfg['classes']),
                        snr_list=np.array(cfg['snr_list_db'], dtype=np.float32))
    
    sz = os.path.getsize(path) / 1024 / 1024
    print(f' 已保存 → {path}')
    print(f' Train:{len(i_tr):,} Test: {len(i_te):,} Size:{sz:.1f} MB')
    return path